<a href="https://colab.research.google.com/github/Ism-ail11/CAPQ-RD/blob/main/capq_rd.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Choose **Runtime → Change runtime type → GPU**, then **Runtime → Run all**. All implementation, real-data loaders, numerical checks, training, compressed-model codecs and result reporting are embedded in this one file. The first code cell writes the embedded script; the second runs it. Dependencies are installed only when missing.

The default full experiment is **CIFAR-100 / ResNet-20 + ten-word Speech Commands v2 / ReLU CNN**, with five independent teacher seeds **11, 23, 37, 53, 71**. It evaluates three actual model-file byte budgets, matched local/static/diagonal/no-gauge controls, uniform quantization, PoT and a weight-only LSQ+KD adaptation. Tables and plots show **mean ± sample SD (ddof=1)** and paired gains. The complete run can take hours on a Colab GPU.


Optional alternative commands, each still using all five seeds: run with --datasets fashionmnist for the full supplementary dataset; --datasets digits for a real-data verification run; or --smoke for reduced real-data CNN integration checks. The default run below uses the full two-task protocol.


## Proposed Method

The candidate contribution is **conditional block allocation after eliminating admissible channel gauges, under exact complete model-file byte constraints**. The module adds nested one-bit mantissa planes, a calibrated PSD sketch, full within-block curvature, conditional cross-block updates, jointly scored upgrade/downgrade exchanges, a consistent gauge optimizer and a calibration-only safeguard.

For fixed code targets and an unconstrained gauge, the quadratic is

Here $v$ is the log-magnitude quantization residual, $B$ maps admissible channel log-scales to weight log-scales, and $H=(1-\eta)\operatorname{diag}(\alpha)+V^TV$ is a fixed PSD calibration metric. $V$ is the normalized output-Jacobian sketch; the local and coupled controls share their chosen metric and initialization.

$$E(v)=\min_s(v+Bs)^T H(v+Bs)=v^T Qv,$$
$$Q=H-HB(B^THB)^{-1}B^TH.$$

A block upgrade $\mu_g$ has gain $2v^TQ\mu_g-\mu_g^TQ\mu_g$. For disjoint blocks, accepting $g$ changes $h$'s gain by $2(j_h^TL^{-1}j_g-z_h^Tz_g)$, with $L=B^THB$, $j_g=B^TH\mu_g$ and $z_g=V\mu_g$. The code verifies these identities on graphs built from trained real-data models. Bounded finite-lattice optimization and greedy allocation remain approximations.

The codec checks every accepted allocation and every final refined file against its byte cap. The safeguard selects proposals using training calibration teacher KL; validation selects refinement checkpoints. Held-out test results make neither decision. All coupled ablations use the safeguard and its fallback choice is logged.

Channel equalization, power-of-two codebooks, sensitivity metrics, learned steps and Schur complements are established ingredients. The candidate novelty is the specific interacting allocation rule and exact codec feasibility, whose value needs support from the controls. See [CLE](https://arxiv.org/abs/1906.04721), [APoT](https://arxiv.org/abs/1909.13144), [HAWQ](https://arxiv.org/abs/1905.03696), [LSQ](https://arxiv.org/abs/1902.08153), [BRECQ](https://arxiv.org/abs/2102.05426), [GPTQ](https://arxiv.org/abs/2210.17323) and [contraction-gauge preconditioning](https://arxiv.org/abs/2607.18745).


## Measured CPU development checks — separate from the default CNN run

These are previously executed measurements of this exact embedded source. The full Fashion-MNIST MLP and UCI digits verification runs each completed 17 conditions × 5 seeds. Reduced CIFAR-100 and Speech Commands integration runs each completed 4 conditions × 5 seeds; they verify execution and the math/codec but do not establish full-benchmark performance. The default full CNN run above has not been executed in this development environment.

| Dataset | Byte cap from | Local accuracy (%) | CAPQ accuracy (%) | Paired gain (pp) | CAPQ stored bits/parameter | Local fallbacks |
|---|---|---:|---:|---:|---:|---:|
| Fashion-MNIST | Uniform 4-bit file | 89.69 ± 0.15 | 89.65 ± 0.16 | -0.040 ± 0.126 | 4.113 ± 0.000 | 3/5 |
| Fashion-MNIST | Uniform 5-bit file | 89.59 ± 0.16 | 89.64 ± 0.18 | +0.052 ± 0.070 | 5.111 ± 0.000 | 2/5 |
| Fashion-MNIST | Uniform 6-bit file | 89.60 ± 0.16 | 89.61 ± 0.13 | +0.004 ± 0.036 | 6.109 ± 0.000 | 3/5 |
| UCI digits | Uniform 4-bit file | 96.56 ± 0.61 | 96.67 ± 0.56 | +0.111 ± 0.248 | 4.862 ± 0.000 | 4/5 |
| UCI digits | Uniform 5-bit file | 96.72 ± 0.46 | 96.67 ± 0.52 | -0.056 ± 0.124 | 5.851 ± 0.000 | 4/5 |
| UCI digits | Uniform 6-bit file | 96.67 ± 0.56 | 96.67 ± 0.56 | +0.000 ± 0.000 | 6.838 ± 0.001 | 5/5 |

Local fallbacks count seeds where the safeguard chose the local proposal. Gains use matched teachers and seeds. The measured effects are small; these results do not establish decisive superiority. Every table and error bar uses sample SD across all five seeds. Complete raw development rows, protocols, dataset manifests and numerical-check records are embedded in notebook metadata under capq.measured_development.


In [ ]:
%%writefile capq_spl.py
#!/usr/bin/env python3
"""CAPQ-2.3: calibrated quotient allocation under exact model-byte budgets.

COLAB: run all cells in capq_spl.ipynb, or upload this script and run
    %run capq_spl.py

Default SPL protocol: CIFAR-100 / ResNet-20 and ten-word Speech Commands v2
/ ReLU CNN. ALL conditions use five independent training seeds: 11,23,37,53,71.
Tables/figures show mean +/- SAMPLE SD (ddof=1), paired same-teacher controls,
pre-refinement accuracy, teacher KL (temperature 2), centered-logit NMSE,
and actual stored bytes. No fabricated results,
synthetic classification examples, seed deletion, or test-set decisions.
The complete default CNN protocol can take hours on a Colab GPU.
"""
from __future__ import annotations

import argparse
import contextlib
import copy
import csv
import dataclasses
import gzip
import hashlib
import importlib.util
import io
import json
import math
import os
from pathlib import Path
import pickle
import random
import struct
import subprocess
import sys
import tarfile
import time
import urllib.request
import wave
import zlib
from dataclasses import dataclass
from typing import Any


def ensure_dependencies() -> None:
    packages = {"numpy": "numpy", "scipy": "scipy", "pandas": "pandas",
                "matplotlib": "matplotlib", "sklearn": "scikit-learn", "torch": "torch", "jinja2":"jinja2"}
    missing = [p for m, p in packages.items() if importlib.util.find_spec(m) is None]
    if missing:
        print("Installing missing dependencies:", ", ".join(missing), flush=True)
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])


ensure_dependencies()
import numpy as np
import pandas as pd
import scipy.linalg as sla
import scipy.optimize as sopt
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

VERSION = "CAPQ-2.3"
SOURCE_SHA256 = hashlib.sha256(Path(__file__).read_bytes()).hexdigest()
SEEDS = (11, 23, 37, 53, 71)
SPLIT_SEED = 2026
COEFF = np.array([0.5, 0.75, 0.875, 0.9375, 1.0, 1.0625, 1.125, 1.25, 1.5], np.float32)
MIDPOINTS = (COEFF[:-1] + COEFF[1:]) / 2
TERM_BITS = 4  # 9 coefficients; do not incorrectly count these as 3 bits.
MAIN_METHODS = ("CLE+local", "Sensitivity", "CAPQ", "CAPQ-compact")
ABLATIONS = ("CAPQ-no-gauge", "CAPQ-no-sensitivity")
MAGIC = b"CAPQv1\x00\x00"
MAGIC2 = b"CAPQv2\x00\x00"
RD_MAIN_METHODS = ("Local-RD", "CAPQ-RD")
RD_ABLATIONS = ("Static-RD", "Diagonal-RD", "RD-no-gauge")


@dataclass
class Config:
    protocol: str = "rd"
    datasets: tuple[str, ...] = ("cifar100", "speechcommands")
    fractions: tuple[float, ...] = (0.02, 0.10, 0.25)
    uniform_bits: tuple[int, ...] = (4, 5, 6)
    exponent_bits: int = 3  # one sign bit + three exponent bits = four base bits
    gauge_steps: int = 8
    allocation_rounds: int = 4
    gauge_bound: float = 1.5
    compact_block_size: int = 16
    calibration_size: int = 768
    selection_size: int = 1024
    sketch_batches: int = 8
    sketch_probes: int = 2
    refinement_steps: int = 240
    refinement_lr: float = 0.0003
    kd_temperature: float = 2.0
    kd_weight: float = 0.8
    eval_interval: int = 40
    cifar_epochs: int = 80
    speech_epochs: int = 35
    fashion_epochs: int = 25
    digits_epochs: int = 40
    batch_size: int = 128
    workers: int = 0  # portable in Colab and restricted CPU hosts; --workers 2 is optional
    cpu_threads: int = 2
    smoke: bool = False
    data_root: str = "capq_data"
    out: str = "capq_outputs"
    device: str = "auto"
    skip_self_tests: bool = False
    methods: tuple[str, ...] = ()
    teacher_cache: str = ""  # optional, explicitly selected trusted CAPQ run
    rd_batch: int = 48
    rd_shortlist: int = 128
    rd_passes: int = 2
    rd_max_depth: int = 3
    rd_swaps: int = 24
    rd_sketch_mix: float = 0.25
    rd_calibration_guard: bool = True
    rd_adaptive_metric: bool = True

    def fingerprint(self) -> str:
        d = dataclasses.asdict(self)
        for k in ("out", "data_root", "device", "cpu_threads", "skip_self_tests", "teacher_cache"):
            d.pop(k)
        d.update(version=VERSION, seeds=SEEDS, split_seed=SPLIT_SEED,
                 torch=torch.__version__, numpy=np.__version__, source_sha256=SOURCE_SHA256)
        return hashlib.sha256(json.dumps(d, sort_keys=True).encode()).hexdigest()[:16]


def log(message: str) -> None:
    print(time.strftime("[%H:%M:%S] ") + message, flush=True)


def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    torch.use_deterministic_algorithms(True, warn_only=True)


def atomic_json(path: Path, value: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(value, indent=2, sort_keys=True), encoding="utf-8")
    os.replace(tmp, path)


def atomic_torch_save(value: Any, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    torch.save(value, tmp)
    os.replace(tmp, path)


def local_torch_load(path: Path, device: str | torch.device = "cpu") -> Any:
    # Only checkpoints written by this script; not arbitrary downloaded pickle files.
    return torch.load(path, map_location=device, weights_only=False)


def download(url: str, destination: Path, md5: str | None = None, sha256: str | None = None) -> None:
    if md5 is not None and (len(md5) != 32 or any(c not in "0123456789abcdef" for c in md5)):
        raise ValueError("A dataset MD5 must contain exactly 32 hexadecimal characters")
    if sha256 is not None and (len(sha256) != 64 or any(c not in "0123456789abcdef" for c in sha256)):
        raise ValueError("A dataset SHA256 must contain exactly 64 hexadecimal characters")
    destination.parent.mkdir(parents=True, exist_ok=True)
    def valid() -> bool:
        if not destination.exists():
            return False
        expected_hash = sha256 or md5
        if expected_hash is None:
            return destination.stat().st_size > 0
        h = hashlib.sha256() if sha256 else hashlib.md5()
        with destination.open("rb") as f:
            for block in iter(lambda: f.read(8 * 1024 * 1024), b""):
                h.update(block)
        return h.hexdigest() == expected_hash
    if valid():
        return
    partial = destination.with_suffix(destination.suffix + ".part")
    last_error = None
    for attempt in range(3):
        try:
            offset = partial.stat().st_size if partial.exists() else 0
            request = urllib.request.Request(url, headers={"User-Agent": "CAPQ/1.0",
                                                          **({"Range": f"bytes={offset}-"} if offset else {})})
            with urllib.request.urlopen(request, timeout=60) as response:
                resumed = offset > 0 and response.status == 206
                if not resumed:
                    offset = 0
                size = int(response.headers.get("Content-Length", 0)) + offset
                next_report = time.monotonic()
                with partial.open("ab" if resumed else "wb") as f:
                    for block in iter(lambda: response.read(4 * 1024 * 1024), b""):
                        f.write(block)
                        offset += len(block)
                        if time.monotonic() >= next_report:
                            log(f"Downloading {destination.name}: {offset / 2**20:.0f} / {size / 2**20:.0f} MiB")
                            next_report = time.monotonic() + 20
                if size and offset != size:
                    raise IOError(f"Incomplete download: {offset} of {size} expected bytes")
            os.replace(partial, destination)
            if not valid():
                destination.unlink(missing_ok=True)
                raise RuntimeError(f"Checksum failed for {destination.name}")
            return
        except Exception as exc:
            last_error = exc
            log(f"Download attempt {attempt + 1}/3 failed: {exc}")
    raise RuntimeError(f"Could not download real dataset from {url}: {last_error}")


def balanced_indices(labels: np.ndarray, count: int, seed: int) -> np.ndarray:
    count = min(count, len(labels))
    rng = np.random.default_rng(seed)
    groups = [rng.permutation(np.flatnonzero(labels == c)).tolist() for c in np.unique(labels)]
    chosen: list[int] = []
    while len(chosen) < count:
        for g in groups:
            if g and len(chosen) < count:
                chosen.append(g.pop())
    return np.array(chosen, np.int64)


class ArrayDataset(Dataset):
    def __init__(self, x: np.ndarray, y: np.ndarray, kind: str, augment: bool = False,
                 mean: float = 0.0, std: float = 1.0):
        self.x, self.y, self.kind, self.augment = x, y.astype(np.int64), kind, augment
        self.mean, self.std = mean, std

    def __len__(self) -> int:
        return len(self.y)

    def __getitem__(self, i: int) -> tuple[torch.Tensor, int]:
        # Cast FP16 audio-cache entries in NumPy first. This is also portable to
        # CPU runtimes whose torch FP16 conversion needs unavailable cpuinfo.
        a = torch.from_numpy(np.array(self.x[i], dtype=np.float32, copy=True))
        if self.kind == "cifar100":
            a = a.permute(2, 0, 1) / 255
            if self.augment:
                a = F.pad(a, (4, 4, 4, 4), mode="reflect")
                h, w = torch.randint(0, 9, (2,)).tolist()
                a = a[:, h:h + 32, w:w + 32]
                if torch.rand(()) < 0.5:
                    a = a.flip(-1)
            mean = a.new_tensor([0.5071, 0.4867, 0.4408])[:, None, None]
            std = a.new_tensor([0.2675, 0.2565, 0.2761])[:, None, None]
            a = (a - mean) / std
        elif self.kind == "fashionmnist":
            a = (a.unsqueeze(0) / 255 - 0.2860) / 0.3530
        elif self.kind == "digits":
            a = a / 16
        elif self.kind == "speechcommands":
            a = (a - self.mean) / self.std
        else:
            raise ValueError(self.kind)
        return a, int(self.y[i])


@dataclass
class DataBundle:
    train: Dataset
    clean_train: Dataset
    val: Dataset
    test: Dataset
    labels: np.ndarray
    val_labels: np.ndarray
    architecture: dict[str, Any]
    dataset_info: dict[str, Any]


def split_arrays(x: np.ndarray, y: np.ndarray, xt: np.ndarray, yt: np.ndarray,
                 kind: str, architecture: dict, cfg: Config) -> DataBundle:
    tr, va = train_test_split(np.arange(len(y)), test_size=0.1, random_state=SPLIT_SEED, stratify=y)
    if cfg.smoke:
        tr = tr[balanced_indices(y[tr], min(1500, len(tr)), SPLIT_SEED)]
        va = va[balanced_indices(y[va], min(400, len(va)), SPLIT_SEED)]
        ti = balanced_indices(yt, min(500, len(yt)), SPLIT_SEED)
        xt, yt = xt[ti], yt[ti]
    return DataBundle(ArrayDataset(x[tr], y[tr], kind, augment=(kind == "cifar100")),
                      ArrayDataset(x[tr], y[tr], kind), ArrayDataset(x[va], y[va], kind),
                      ArrayDataset(xt, yt, kind), y[tr], y[va], architecture,
                      dict(dataset=kind, train=len(tr), validation=len(va), test=len(yt),
                           split_seed=SPLIT_SEED, reduced_data=cfg.smoke))


def load_cifar100(cfg: Config) -> DataBundle:
    root = Path(cfg.data_root) / "cifar100"
    archive = root / "cifar-100-python.tar.gz"
    download("https://www.cs.toronto.edu/~kriz/cifar-100-python.tar.gz", archive,
             "eb9058c3a382ffc7106e4002c42a8d85")
    # Verified official archive; never unpickle an unverified user-provided file.
    with tarfile.open(archive, "r:gz") as f:
        train_file = f.extractfile("cifar-100-python/train")
        test_file = f.extractfile("cifar-100-python/test")
        if train_file is None or test_file is None:
            raise RuntimeError("Official CIFAR-100 archive is missing data members")
        tr = pickle.load(train_file, encoding="bytes")
        te = pickle.load(test_file, encoding="bytes")
    x = tr[b"data"].reshape(-1, 3, 32, 32).transpose(0, 2, 3, 1)
    xt = te[b"data"].reshape(-1, 3, 32, 32).transpose(0, 2, 3, 1)
    arch = dict(name="resnet20", classes=100, width=16, blocks_per_stage=3)
    return split_arrays(x, np.array(tr[b"fine_labels"]), xt, np.array(te[b"fine_labels"]),
                        "cifar100", arch, cfg)


def read_idx(path: Path) -> np.ndarray:
    with gzip.open(path, "rb") as f:
        magic, n = struct.unpack(">II", f.read(8))
        if magic == 2051:
            h, w = struct.unpack(">II", f.read(8))
            return np.frombuffer(f.read(), np.uint8).reshape(n, h, w).copy()
        if magic == 2049:
            return np.frombuffer(f.read(), np.uint8).copy()
    raise ValueError(f"Invalid IDX file {path}")


def load_fashion(cfg: Config) -> DataBundle:
    root = Path(cfg.data_root) / "fashionmnist"
    files = [
        ("train-images-idx3-ubyte.gz", "8d4fb7e6c68d591d4c3dfef9ec88bf0d"),
        ("train-labels-idx1-ubyte.gz", "25c81989df183df01b3e8a0aad5dffbe"),
        ("t10k-images-idx3-ubyte.gz", "bef4ecab320f06d8554ea6380940ec79"),
        ("t10k-labels-idx1-ubyte.gz", "bb300cfdad3c16e7a12a480ee83cd310")]
    for name, digest in files:
        download("https://raw.githubusercontent.com/zalandoresearch/fashion-mnist/master/data/fashion/" + name,
                 root / name, digest)
    x, y, xt, yt = [read_idx(root / name) for name, _ in files]
    return split_arrays(x, y, xt, yt, "fashionmnist",
                        dict(name="mlp", input_dim=784, widths=[256, 128], classes=10), cfg)


def load_real_digits(cfg: Config) -> DataBundle:
    d = load_digits()
    tr, te = train_test_split(np.arange(len(d.target)), test_size=0.2,
                             random_state=SPLIT_SEED, stratify=d.target)
    return split_arrays(d.data[tr].astype(np.float32), d.target[tr],
                        d.data[te].astype(np.float32), d.target[te], "digits",
                        dict(name="mlp", input_dim=64, widths=[128, 64], classes=10), cfg)


SPEECH_WORDS = ("yes", "no", "up", "down", "left", "right", "on", "off", "stop", "go")


def mel_filter(sample_rate: int = 16000, n_fft: int = 512, n_mels: int = 40) -> torch.Tensor:
    hz_to_mel = lambda x: 2595 * np.log10(1 + np.asarray(x) / 700)
    mel_to_hz = lambda x: 700 * (10 ** (np.asarray(x) / 2595) - 1)
    centers = mel_to_hz(np.linspace(hz_to_mel(20), hz_to_mel(7600), n_mels + 2))
    hz = np.linspace(0, sample_rate / 2, n_fft // 2 + 1)
    filters = np.maximum(0, np.minimum((hz[None, :] - centers[:-2, None]) /
                                     (centers[1:-1, None] - centers[:-2, None]),
                                     (centers[2:, None] - hz[None, :]) /
                                     (centers[2:, None] - centers[1:-1, None])))
    return torch.from_numpy(filters.astype(np.float32))


def read_wav(path: Path) -> np.ndarray:
    with wave.open(str(path), "rb") as f:
        if (f.getframerate(), f.getnchannels(), f.getsampwidth()) != (16000, 1, 2):
            raise ValueError(f"Expected 16 kHz mono PCM16: {path}")
        a = np.frombuffer(f.readframes(f.getnframes()), dtype="<i2").astype(np.float32) / 32768
    result = np.zeros(16000, np.float32)
    result[:min(16000, len(a))] = a[:16000]
    return result


def load_speech(cfg: Config) -> DataBundle:
    root = Path(cfg.data_root) / "speechcommands_v2"
    extracted = root / "wav"
    ready = extracted / "EXTRACTION_COMPLETE.json"
    if not ready.exists():
        archive = root / "speech_commands_v0.02.tar.gz"
        # Official TensorFlow Datasets checksums.tsv, Speech Commands v0.02.
        download("https://storage.googleapis.com/download.tensorflow.org/data/speech_commands_v0.02.tar.gz", archive,
                 sha256="af14739ee7dc311471de98f5f9d2c9191b18aedfe957f4a6ff791c709868ff58")
        extracted.mkdir(parents=True, exist_ok=True)
        log("Extracting official split lists and the ten real-word folders")
        with tarfile.open(archive, "r:gz") as tar:
            for member in tar:
                rel = member.name.removeprefix("./")
                parts = Path(rel).parts
                if not member.isfile() or not parts or ".." in parts or Path(rel).is_absolute():
                    continue
                wanted = rel in ("validation_list.txt", "testing_list.txt") or (
                    parts[0] in SPEECH_WORDS and rel.endswith(".wav"))
                if wanted:
                    dest = extracted / rel
                    dest.parent.mkdir(parents=True, exist_ok=True)
                    source = tar.extractfile(member)
                    if source is None:
                        raise RuntimeError(f"Cannot extract {rel}")
                    with source, dest.open("wb") as out:
                        while block := source.read(1024 * 1024):
                            out.write(block)
        if not (extracted / "validation_list.txt").exists() or not (extracted / "testing_list.txt").exists():
            raise RuntimeError("Speech Commands archive has no official split lists")
        atomic_json(ready, dict(version=2, words=SPEECH_WORDS))
    val_names = set((extracted / "validation_list.txt").read_text().splitlines())
    test_names = set((extracted / "testing_list.txt").read_text().splitlines())
    paths = sorted(p for word in SPEECH_WORDS for p in (extracted / word).glob("*.wav"))
    if not paths:
        raise RuntimeError("No real Speech Commands waveforms found")
    labels = np.array([SPEECH_WORDS.index(p.parent.name) for p in paths], np.int64)
    split = np.array([2 if p.relative_to(extracted).as_posix() in test_names else
                      1 if p.relative_to(extracted).as_posix() in val_names else 0 for p in paths])
    # Verify actual speaker separation before any training.
    speakers = [{p.name.split("_nohash_")[0] for i, p in enumerate(paths) if split[i] == k} for k in range(3)]
    assert not (speakers[0] & speakers[1] or speakers[0] & speakers[2] or speakers[1] & speakers[2]), "Speaker leakage"
    cache_name = "logmel40_fft512_hop160_v1.npy"
    feature_path = root / cache_name
    cache_meta = root / (cache_name + ".json")
    manifest_hash = hashlib.sha256("\n".join(p.relative_to(extracted).as_posix() for p in paths).encode()).hexdigest()
    expected = dict(manifest_hash=manifest_hash, count=len(paths), shape=[len(paths), 1, 40, 101])
    if not (feature_path.exists() and cache_meta.exists() and json.loads(cache_meta.read_text()) == expected):
        tmp = feature_path.with_suffix(".partial.npy")
        mmap = np.lib.format.open_memmap(tmp, mode="w+", dtype=np.float16, shape=tuple(expected["shape"]))
        filters, window = mel_filter(), torch.hann_window(400)
        log(f"Caching log-mel features for {len(paths)} real waveforms")
        for start in range(0, len(paths), 128):
            audio = torch.from_numpy(np.stack([read_wav(p) for p in paths[start:start + 128]]))
            power = torch.stft(audio, n_fft=512, hop_length=160, win_length=400,
                               window=window, return_complex=True).abs().square()
            features = torch.log((filters[None] @ power).clamp_min(1e-8))
            mmap[start:start + len(audio), 0] = features.numpy().astype(np.float16)
            if start % (128 * 50) == 0:
                log(f"Audio features: {start}/{len(paths)}")
        mmap.flush()
        del mmap
        os.replace(tmp, feature_path)
        atomic_json(cache_meta, expected)
    features = np.load(feature_path, mmap_mode="r")
    ids = [np.flatnonzero(split == k) for k in range(3)]
    if cfg.smoke:
        ids = [a[balanced_indices(labels[a], min(n, len(a)), SPLIT_SEED)]
               for a, n in zip(ids, (1500, 400, 500))]
    # Only the official training set determines normalization statistics.
    total = total_sq = 0.0
    n_values = 0
    for start in range(0, len(ids[0]), 256):
        a = features[ids[0][start:start + 256]].astype(np.float64)
        total += a.sum()
        total_sq += np.square(a).sum()
        n_values += a.size
    mean = total / n_values
    std = math.sqrt(max(total_sq / n_values - mean * mean, 1e-6))
    datasets = [ArrayDataset(features[a], labels[a], "speechcommands", mean=mean, std=std) for a in ids]
    arch = dict(name="speechcnn", widths=[32, 64, 96, 128], classes=10)
    return DataBundle(datasets[0], datasets[0], datasets[1], datasets[2], labels[ids[0]],
                      labels[ids[1]], arch, dict(dataset="speechcommands", version=2,
                      task="10-word classification; silence and unknown are excluded",
                      words=SPEECH_WORDS, train=len(ids[0]), validation=len(ids[1]), test=len(ids[2]),
                      official_speaker_disjoint=True, reduced_data=cfg.smoke,
                      normalization_mean=mean, normalization_std=std))


class MLP(nn.Module):
    def __init__(self, input_dim: int, widths: list[int], classes: int):
        super().__init__()
        sizes = [input_dim, *widths, classes]
        self.linears = nn.ModuleList([nn.Linear(a, b) for a, b in zip(sizes[:-1], sizes[1:])])

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x.flatten(1)
        for layer in self.linears[:-1]:
            x = F.relu(layer(x))
        return self.linears[-1](x)


class ResidualBlock(nn.Module):
    def __init__(self, incoming: int, outgoing: int, stride: int):
        super().__init__()
        self.conv1 = nn.Conv2d(incoming, outgoing, 3, stride, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(outgoing)
        self.conv2 = nn.Conv2d(outgoing, outgoing, 3, 1, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(outgoing)
        self.shortcut = (nn.Sequential(nn.Conv2d(incoming, outgoing, 1, stride, bias=False),
                                       nn.BatchNorm2d(outgoing)) if incoming != outgoing or stride != 1 else nn.Identity())

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        z = F.relu(self.bn1(self.conv1(x)))
        z = self.bn2(self.conv2(z))
        return F.relu(z + self.shortcut(x))


class ResNet20(nn.Module):
    def __init__(self, classes: int = 100, width: int = 16, blocks_per_stage: int = 3):
        super().__init__()
        self.stem = nn.Conv2d(3, width, 3, 1, 1, bias=False)
        self.bn = nn.BatchNorm2d(width)
        blocks = []
        incoming = width
        for stage in range(3):
            outgoing = width * 2 ** stage
            for k in range(blocks_per_stage):
                stride = 2 if stage > 0 and k == 0 else 1
                blocks.append(ResidualBlock(incoming, outgoing, stride))
                incoming = outgoing
        self.blocks = nn.ModuleList(blocks)
        self.fc = nn.Linear(incoming, classes)
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = F.relu(self.bn(self.stem(x)))
        for b in self.blocks:
            x = b(x)
        return self.fc(F.adaptive_avg_pool2d(x, 1).flatten(1))


class SpeechCNN(nn.Module):
    def __init__(self, widths: list[int], classes: int):
        super().__init__()
        sizes = [1, *widths]
        self.convs = nn.ModuleList([nn.Conv2d(a, b, 3, padding=1, bias=False) for a, b in zip(sizes[:-1], sizes[1:])])
        self.bns = nn.ModuleList([nn.BatchNorm2d(b) for b in widths])
        self.fc = nn.Linear(widths[-1], classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        for i, (conv, bn) in enumerate(zip(self.convs, self.bns)):
            x = F.relu(bn(conv(x)))
            if i < 2:
                x = F.max_pool2d(x, 2)
        return self.fc(F.adaptive_avg_pool2d(x, 1).flatten(1))


def build_model(arch: dict) -> nn.Module:
    name = arch["name"]
    args = {k: v for k, v in arch.items() if k not in ("name", "folded")}
    model = {"mlp": MLP, "resnet20": ResNet20, "speechcnn": SpeechCNN}[name](**args)
    if arch.get("folded", False):
        model = fold_batchnorm(model.eval())
    return model


def fuse_conv_bn(conv: nn.Conv2d, bn: nn.BatchNorm2d) -> None:
    if bn.training:
        raise ValueError("BatchNorm must be in eval mode before folding")
    with torch.no_grad():
        a = bn.weight / torch.sqrt(bn.running_var + bn.eps)
        b = conv.bias if conv.bias is not None else torch.zeros_like(bn.running_mean)
        conv.weight.mul_(a.reshape(-1, 1, 1, 1))
        conv.bias = nn.Parameter((b - bn.running_mean) * a + bn.bias)


def fold_batchnorm(model: nn.Module) -> nn.Module:
    result = copy.deepcopy(model).eval()
    if isinstance(result, ResNet20):
        if isinstance(result.bn, nn.BatchNorm2d):
            fuse_conv_bn(result.stem, result.bn)
            result.bn = nn.Identity()
        for block in result.blocks:
            if isinstance(block.bn1, nn.BatchNorm2d):
                fuse_conv_bn(block.conv1, block.bn1)
                block.bn1 = nn.Identity()
                fuse_conv_bn(block.conv2, block.bn2)
                block.bn2 = nn.Identity()
            if isinstance(block.shortcut, nn.Sequential) and isinstance(block.shortcut[1], nn.BatchNorm2d):
                fuse_conv_bn(block.shortcut[0], block.shortcut[1])
                block.shortcut[1] = nn.Identity()
    elif isinstance(result, SpeechCNN):
        for i in range(len(result.convs)):
            if isinstance(result.bns[i], nn.BatchNorm2d):
                fuse_conv_bn(result.convs[i], result.bns[i])
                result.bns[i] = nn.Identity()
    return result


def module_at(model: nn.Module, name: str) -> nn.Module:
    return model.get_submodule(name)


def replace_module(model: nn.Module, name: str, new: nn.Module) -> None:
    parent_name, _, child = name.rpartition(".")
    parent = model.get_submodule(parent_name) if parent_name else model
    setattr(parent, child, new)


@dataclass
class LayerEdges:
    name: str
    shape: tuple[int, ...]
    start: int
    stop: int
    heads: np.ndarray
    tails: np.ndarray


class WeightGraph:
    """Admissible channel gauges; identity-residual endpoints share node IDs."""
    def __init__(self, model: nn.Module, arch: dict, alpha: np.ndarray | None = None):
        self.model, self.arch = model, arch
        next_id = 0
        def nodes(n: int) -> np.ndarray:
            nonlocal next_id
            a = np.arange(next_id, next_id + n, dtype=np.int32)
            next_id += n
            return a
        anchor = lambda n: np.full(n, -1, np.int32)
        bindings = []
        if isinstance(model, MLP):
            previous = anchor(model.linears[0].in_features)
            for i, m in enumerate(model.linears):
                head = nodes(m.out_features) if i < len(model.linears) - 1 else anchor(m.out_features)
                bindings.append((f"linears.{i}", head, previous))
                previous = head
        elif isinstance(model, SpeechCNN):
            previous = anchor(1)
            for i, m in enumerate(model.convs):
                head = nodes(m.out_channels)
                bindings.append((f"convs.{i}", head, previous))
                previous = head
            bindings.append(("fc", anchor(model.fc.out_features), previous))
        elif isinstance(model, ResNet20):
            width, repetitions = arch["width"], arch["blocks_per_stage"]
            stage_nodes = [nodes(width * 2 ** i) for i in range(3)]
            bindings.append(("stem", stage_nodes[0], anchor(3)))
            previous = stage_nodes[0]
            for i, block in enumerate(model.blocks):
                stage = i // repetitions
                head = stage_nodes[stage]
                inner = nodes(block.conv1.out_channels)
                bindings.append((f"blocks.{i}.conv1", inner, previous))
                bindings.append((f"blocks.{i}.conv2", head, inner))
                if isinstance(block.shortcut, nn.Sequential):
                    bindings.append((f"blocks.{i}.shortcut.0", head, previous))
                elif not np.array_equal(head, previous):
                    raise AssertionError("An identity shortcut cannot have independent endpoint scales")
                previous = head
            bindings.append(("fc", anchor(model.fc.out_features), previous))
        else:
            raise TypeError(f"No graph adapter for {type(model)}")
        self.n_nodes = next_id
        self.layers: list[LayerEdges] = []
        weights, heads_all, tails_all = [], [], []
        start = 0
        for name, heads, tails in bindings:
            m = module_at(model, name)
            w = m.weight.detach().cpu().numpy().astype(np.float64)
            if isinstance(m, nn.Conv2d) and m.groups != 1:
                raise ValueError("This experiment uses full convolutions, not grouped convolutions")
            shape = w.shape
            h = np.broadcast_to(heads.reshape((-1,) + (1,) * (w.ndim - 1)), shape).reshape(-1).copy()
            t = np.broadcast_to(tails.reshape((1, -1) + (1,) * (w.ndim - 2)), shape).reshape(-1).copy()
            weights.append(w.reshape(-1)); heads_all.append(h); tails_all.append(t)
            self.layers.append(LayerEdges(name, shape, start, start + w.size, heads, tails))
            start += w.size
        self.weights = np.concatenate(weights)
        self.head = np.concatenate(heads_all)
        self.tail = np.concatenate(tails_all)
        self.magnitude = np.abs(self.weights)
        self.x = np.log2(np.maximum(self.magnitude, 1e-30))
        self.valid = self.magnitude > 1e-12
        if alpha is None:
            alpha = np.maximum(np.square(self.weights), 1e-20)
        if alpha.shape != self.weights.shape or not np.all(np.isfinite(alpha)):
            raise ValueError("Invalid graph sensitivity vector")
        self.alpha = np.asarray(alpha, np.float64).copy()
        self.alpha[~self.valid] = 0
        positive = self.alpha[self.valid]
        scale = float(np.median(positive)) if len(positive) else 1
        self.alpha /= max(scale, 1e-30)
        self.alpha[self.valid] = np.clip(self.alpha[self.valid], 1e-4, 1e4)
        self._make_laplacian()

    @property
    def n_weights(self) -> int:
        return len(self.weights)

    def shift(self, s: np.ndarray) -> np.ndarray:
        extended = np.concatenate([s, [0.0]])
        return extended[self.head] - extended[self.tail]

    def rhs(self, v: np.ndarray, weights: np.ndarray | None = None) -> np.ndarray:
        a = (self.alpha if weights is None else weights) * v
        h, t = self.head >= 0, self.tail >= 0
        return (np.bincount(self.head[h], weights=a[h], minlength=self.n_nodes) -
                np.bincount(self.tail[t], weights=a[t], minlength=self.n_nodes))

    def laplacian(self, weights: np.ndarray) -> np.ndarray:
        """B^T diag(weights) B without constructing the huge edge incidence B."""
        L = np.zeros((self.n_nodes, self.n_nodes), np.float64)
        h, t = self.head >= 0, self.tail >= 0
        np.add.at(L, (self.head[h], self.head[h]), weights[h])
        np.add.at(L, (self.tail[t], self.tail[t]), weights[t])
        both = h & t
        np.add.at(L, (self.head[both], self.tail[both]), -weights[both])
        np.add.at(L, (self.tail[both], self.head[both]), -weights[both])
        return L

    def _make_laplacian(self) -> None:
        n = self.n_nodes
        L = self.laplacian(self.alpha)
        # Anchored graphs are SPD; only add a tiny numerical ridge if necessary.
        ridge = max(float(np.max(np.diag(L), initial=0)), 1) * 1e-12
        try:
            factor = sla.cho_factor(L, check_finite=False)
            self.numerical_ridge = 0.0
        except np.linalg.LinAlgError:
            L += np.eye(n) * ridge
            factor = sla.cho_factor(L, check_finite=False)
            self.numerical_ridge = ridge
        self.L, self.factor = L, factor
        inv = sla.cho_solve(factor, np.eye(n), check_finite=False)
        pad = np.pad(inv, ((0, 1), (0, 1)))
        resistance = pad[self.head, self.head] + pad[self.tail, self.tail] - 2 * pad[self.head, self.tail]
        self.cycle_curvature = self.alpha * np.clip(1 - self.alpha * resistance, 1e-8, 1)

    def project_residual(self, v: np.ndarray) -> np.ndarray:
        s = -sla.cho_solve(self.factor, self.rhs(v), check_finite=False)
        return v + self.shift(s)

    def solve_scales(self, target: np.ndarray, bound: float, warm: np.ndarray) -> np.ndarray:
        b = self.rhs(self.x - target)
        solution = -sla.cho_solve(self.factor, b, check_finite=False)
        if np.max(np.abs(solution), initial=0) <= bound:
            return solution
        def fun(s: np.ndarray) -> tuple[float, np.ndarray]:
            gradient = self.L @ s + b
            return float(0.5 * s @ (self.L @ s) + b @ s), gradient
        result = sopt.minimize(fun, np.clip(warm, -bound, bound), jac=True, method="L-BFGS-B",
                               bounds=[(-bound, bound)] * self.n_nodes,
                               options=dict(maxiter=150, ftol=1e-12, gtol=1e-7))
        if not np.all(np.isfinite(result.x)):
            raise FloatingPointError("Non-finite bounded graph solve")
        return result.x

    def apply(self, s: np.ndarray) -> nn.Module:
        result = copy.deepcopy(self.model)
        scale = np.concatenate([np.exp2(s), [1.0]])
        with torch.no_grad():
            for layer in self.layers:
                m = module_at(result, layer.name)
                incoming = torch.as_tensor(scale[layer.tails], device=m.weight.device, dtype=m.weight.dtype)
                outgoing = torch.as_tensor(scale[layer.heads], device=m.weight.device, dtype=m.weight.dtype)
                m.weight.mul_((outgoing[:, None] / incoming[None, :]).reshape(
                    m.weight.shape[0], m.weight.shape[1], *([1] * (m.weight.ndim - 2))))
                if m.bias is not None:
                    m.bias.mul_(outgoing)
        return result

    def cle_scales(self, bound: float) -> np.ndarray:
        # Range-equalization control, respecting exactly the same residual ties.
        s = np.zeros(self.n_nodes, np.float64)
        valid = self.valid
        for _ in range(12):
            x = self.x + self.shift(s)
            incoming = np.full(self.n_nodes, -np.inf)
            outgoing = np.full(self.n_nodes, -np.inf)
            h, t = (self.head >= 0) & valid, (self.tail >= 0) & valid
            np.maximum.at(incoming, self.head[h], x[h])
            np.maximum.at(outgoing, self.tail[t], x[t])
            delta = np.where(np.isfinite(incoming) & np.isfinite(outgoing), (outgoing - incoming) / 2, 0)
            s = np.clip(s + 0.5 * delta, -bound, bound)
        return s


def base_codes(magnitude: np.ndarray, min_exp: int, exp_bits: int) -> tuple[np.ndarray, np.ndarray]:
    maximum = min_exp + (2 ** exp_bits - 2)
    x = np.log2(np.maximum(magnitude, 1e-30))
    lo = np.clip(np.floor(x), min_exp, maximum).astype(np.int32)
    hi = np.clip(lo + 1, min_exp, maximum)
    low, high = np.exp2(lo), np.exp2(hi)
    exponent = np.where(np.abs(magnitude - low) <= np.abs(magnitude - high), lo, hi)
    code = (exponent - min_exp + 1).astype(np.uint8)
    value = np.exp2(exponent)
    zero = magnitude < 0.5 * 2.0 ** min_exp
    code[zero], value[zero] = 0, 0
    return code, value


def dyadic_codebook(min_exp: int, exp_bits: int) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Nearest search over ALL legal exponent/term pairs, including zero.

    Duplicate values prefer a single-term representation. A selected weight may
    use a different exponent from its nearest single-term value. This matters
    at the lower/upper range boundaries and is also used by the STE projector.
    """
    entries = [(0.0, 0, 4)]
    for code in range(1, 2 ** exp_bits):
        for tag, coefficient in enumerate(COEFF):
            entries.append((2.0 ** (min_exp + code - 1) * float(coefficient), code, tag))
    entries.sort(key=lambda e: (e[0], e[2] != 4, e[2] != 0, e[1], e[2]))
    unique = {}
    for value, code, tag in entries:
        unique.setdefault(value, (code, tag))
    values = np.array(list(unique), np.float64)
    codes = np.array([unique[v][0] for v in values], np.uint8)
    tags = np.array([unique[v][1] for v in values], np.uint8)
    return values, codes, tags


def nearest_dyadic(magnitude: np.ndarray, min_exp: int, exp_bits: int) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    values, codes, tags = dyadic_codebook(min_exp, exp_bits)
    ids = np.searchsorted((values[:-1] + values[1:]) / 2, magnitude, side="left")
    return codes[ids], tags[ids], values[ids]


def choose_window(magnitude: np.ndarray, alpha: np.ndarray, exp_bits: int,
                  mask: np.ndarray | None = None) -> int:
    valid = magnitude > 1e-12
    if not np.any(valid):
        return -8
    x = np.log2(magnitude[valid])
    highs = np.floor(np.quantile(x, [0.97, 0.99, 0.999, 1.0])).astype(int)
    candidates = sorted({int(h - (2 ** exp_bits - 2) + shift) for h in highs for shift in (-1, 0, 1)})
    best = (float("inf"), candidates[0])
    for minimum in candidates:
        _, value = base_codes(magnitude, minimum, exp_bits)
        if mask is not None and np.any(mask):
            value[mask] = nearest_dyadic(magnitude[mask], minimum, exp_bits)[2]
        ratio = np.divide(value, magnitude, out=np.zeros_like(value), where=valid)
        cost = float(np.dot(alpha[valid], np.square(ratio[valid] - 1)))
        if cost < best[0]:
            best = (cost, minimum)
    return best[1]


def pot_spec(weight: np.ndarray, alpha: np.ndarray, mask: np.ndarray,
             exp_bits: int, min_exp: int | None = None) -> dict:
    flat = weight.reshape(-1).astype(np.float64)
    magnitude = np.abs(flat)
    if min_exp is None:
        min_exp = choose_window(magnitude, alpha, exp_bits, mask)
    code, base = base_codes(magnitude, min_exp, exp_bits)
    tag = np.full(len(flat), 4, np.uint8)
    if np.any(mask):
        code[mask], tag[mask], base[mask] = nearest_dyadic(magnitude[mask], min_exp, exp_bits)
    value = (np.sign(flat) * base).astype(np.float32)
    return dict(kind="pot", shape=list(weight.shape), exponent_bits=exp_bits, min_exp=int(min_exp),
                base_code=code, sign=flat < 0, mask=mask.copy(), tag=tag,
                value=value.reshape(weight.shape))


def graph_specs(graph: WeightGraph, s: np.ndarray, mask: np.ndarray, bits: int,
                windows: dict[str, int] | None = None) -> tuple[dict, np.ndarray, float]:
    weight = graph.weights * np.exp2(graph.shift(s))
    specs, target = {}, np.empty_like(weight)
    error = 0.0
    for layer in graph.layers:
        a = slice(layer.start, layer.stop)
        minimum = None if windows is None else windows[layer.name]
        spec = pot_spec(weight[a].reshape(layer.shape), graph.alpha[a], mask[a], bits, minimum)
        specs[layer.name] = spec
        magnitude = np.abs(spec["value"].reshape(-1).astype(np.float64))
        target[a] = np.log2(np.where(magnitude > 0, magnitude, 2.0 ** spec["min_exp"]))
        ratio = np.divide(magnitude, np.abs(weight[a]), out=np.zeros_like(magnitude), where=graph.valid[a])
        error += float(np.dot(graph.alpha[a], np.square(ratio - 1)))
    return specs, target, error / max(float(graph.alpha.sum()), 1e-30)


def optimize_gauge(graph: WeightGraph, mask: np.ndarray, cfg: Config,
                   initial: np.ndarray | None = None, steps: int | None = None) -> tuple[np.ndarray, list[float]]:
    s = np.zeros(graph.n_nodes) if initial is None else initial.copy()
    specs, target, cost = graph_specs(graph, s, mask, cfg.exponent_bits)
    history = [cost]
    for _ in range(cfg.gauge_steps if steps is None else steps):
        proposal = graph.solve_scales(target, cfg.gauge_bound, s)
        # A Gauss-Newton step for the actual relative objective with the current
        # quantized values held fixed; all proposals are re-quantized and checked.
        windows = {name: spec["min_exp"] for name, spec in specs.items()}
        q = np.concatenate([np.abs(specs[layer.name]["value"].reshape(-1)) for layer in graph.layers])
        magnitude = graph.magnitude * np.exp2(graph.shift(s))
        ratio = np.divide(q, magnitude, out=np.zeros_like(graph.x), where=graph.valid)
        H = graph.laplacian(graph.alpha * ratio**2)
        H.flat[::graph.n_nodes + 1] += max(float(np.max(np.diag(H), initial=0)), 1) * 1e-9
        rhs = graph.rhs(ratio * (ratio - 1)) / math.log(2)
        delta = sla.solve(H, rhs, assume_a="pos", check_finite=False)
        nonlinear = np.clip(s + delta, -cfg.gauge_bound, cfg.gauge_bound)
        best = (cost, s, target)
        for direction in (proposal, nonlinear):
            for fraction in (1.0, 0.5, 0.25, 0.125):
                candidate = s + fraction * (direction - s)
                _, next_target, next_cost = graph_specs(graph, candidate, mask, cfg.exponent_bits, windows)
                if next_cost < best[0] - 1e-12:
                    best = next_cost, candidate, next_target
        if best[0] >= cost - 1e-12:
            break
        cost, s, target = best
        next_specs, next_target, new_cost = graph_specs(graph, s, mask, cfg.exponent_bits)
        if new_cost <= cost:
            specs, target, cost = next_specs, next_target, new_cost
        else:
            # Quantile-generated candidates can omit the old window after a
            # scale move. Retain it whenever it is better than those candidates.
            specs, target, cost = graph_specs(graph, s, mask, cfg.exponent_bits, windows)
        history.append(cost)
    assert np.all(np.diff(history) <= 1e-10), "The graph line search must not increase its objective"
    return s, history


def allocation_scores(graph: WeightGraph, s: np.ndarray, cfg: Config,
                      cycle_method: bool) -> tuple[np.ndarray, np.ndarray]:
    """Counterfactual gain from one vs two terms, at a common finite window.

    The cycle score eliminates B s from the fixed-code quadratic. Its candidate
    change must be an actually representable nearest two-term value. Returning
    the exact relative gain allows a monotone safeguard and a strong baseline.
    """
    weight = graph.weights * np.exp2(graph.shift(s))
    target = np.empty_like(weight)
    mu = np.zeros_like(weight)
    local_gain = np.zeros_like(weight)
    empty = np.zeros(graph.n_weights, bool)
    for layer in graph.layers:
        a = slice(layer.start, layer.stop)
        one = pot_spec(weight[a].reshape(layer.shape), graph.alpha[a], empty[a], cfg.exponent_bits)
        two = pot_spec(weight[a].reshape(layer.shape), graph.alpha[a], ~empty[a], cfg.exponent_bits, one["min_exp"])
        q1 = np.abs(one["value"].reshape(-1).astype(np.float64))
        q2 = np.abs(two["value"].reshape(-1).astype(np.float64))
        mag = np.abs(weight[a])
        r1 = np.divide(q1, mag, out=np.zeros_like(mag), where=graph.valid[a])
        r2 = np.divide(q2, mag, out=np.zeros_like(mag), where=graph.valid[a])
        local_gain[a] = graph.alpha[a] * np.maximum((r1 - 1)**2 - (r2 - 1)**2, 0)
        target[a] = np.log2(np.where(q1 > 0, q1, 2.0 ** one["min_exp"]))
        mu[a] = np.log2(np.where(q2 > 0, q2, 2.0 ** one["min_exp"])) - target[a]
    if cycle_method:
        residual = graph.project_residual(graph.x - target)
        score = np.maximum(2 * graph.alpha * residual * mu - graph.cycle_curvature * mu**2, 0)
        # Zero has no logarithm: use its actual relative gain instead of a
        # fictitious log residual. This also handles a recovered tiny weight.
        for layer in graph.layers:
            a = slice(layer.start, layer.stop)
            z = base_codes(np.abs(weight[a]), choose_window(np.abs(weight[a]), graph.alpha[a],
                                                           cfg.exponent_bits), cfg.exponent_bits)[0] == 0
            part = score[a]
            part[z] = local_gain[a][z]
        score += local_gain * 1e-6  # deterministic, informative tie break
    else:
        score = local_gain.copy()
    score[~graph.valid] = -np.inf
    return score, local_gain


def top_mask(score: np.ndarray, count: int) -> np.ndarray:
    mask = np.zeros(len(score), bool)
    if count:
        mask[np.argpartition(score, -count)[-count:]] = True
    return mask


def compact_mask_proposals(graph: WeightGraph, s: np.ndarray, count: int,
                           cfg: Config) -> list[np.ndarray]:
    """Full BLOCK marginal, including its off-diagonal cycle interactions.

    For a block perturbation mu, gain = 2 r^T A mu - mu^T A mu
    + (B^T A mu)^T L^{-1}(B^T A mu). Each column of J below is
    B^T A mu for one block. This is NOT the sum of edgewise leverage gains.
    Zero codes use the same finite proxy as the graph surrogate; candidate
    masks are compared after actual finite-lattice quantization and regauging.
    Blocks produce compressible exception masks in the SAME byte-counted
    codec. A single partially filled block makes the weight budget exact.
    """
    n, block = graph.n_weights, cfg.compact_block_size
    gid = np.empty(n, np.int32)
    groups = []
    for layer in graph.layers:
        for start in range(layer.start, layer.stop, block):
            stop = min(start + block, layer.stop)
            gid[start:stop] = len(groups)
            groups.append((start, stop))
    sizes = np.array([b-a for a,b in groups], np.int64)
    ng = len(groups)
    weight = graph.weights * np.exp2(graph.shift(s))
    target, mu, physical = np.empty(n), np.empty(n), np.empty(n)
    for layer in graph.layers:
        a = slice(layer.start, layer.stop)
        shape = layer.shape
        one = pot_spec(weight[a].reshape(shape), graph.alpha[a], np.zeros(layer.stop-layer.start, bool),
                       cfg.exponent_bits)
        two = pot_spec(weight[a].reshape(shape), graph.alpha[a], np.ones(layer.stop-layer.start, bool),
                       cfg.exponent_bits, one["min_exp"])
        q1 = np.abs(one["value"].reshape(-1).astype(np.float64))
        q2 = np.abs(two["value"].reshape(-1).astype(np.float64))
        minimum = 2.0 ** one["min_exp"]
        target[a] = np.log2(np.where(q1 > 0, q1, minimum))
        mu[a] = np.log2(np.where(q2 > 0, q2, minimum)) - target[a]
        magnitude = np.maximum(np.abs(weight[a]), 1e-30)
        physical[a] = graph.alpha[a] * np.maximum((q1/magnitude-1)**2-(q2/magnitude-1)**2, 0)
    residual = graph.project_residual(graph.x-target)
    J = np.zeros((graph.n_nodes, ng), np.float64)
    h, t = graph.head >= 0, graph.tail >= 0
    np.add.at(J, (graph.head[h], gid[h]), graph.alpha[h]*mu[h])
    np.add.at(J, (graph.tail[t], gid[t]), -graph.alpha[t]*mu[t])
    solved = sla.cho_solve(graph.factor, J, check_finite=False)
    local_scores = np.bincount(gid, weights=physical, minlength=ng)
    cycle_scores = np.bincount(gid, weights=2*graph.alpha*residual*mu-graph.alpha*mu**2,
                               minlength=ng) + np.sum(J*solved, axis=0)
    cycle_scores = np.maximum(cycle_scores, 0) + 1e-6*local_scores
    candidates = []
    for scores in (cycle_scores, local_scores):
        order = np.argsort(-scores/sizes, kind="stable")
        mask = np.zeros(n, bool)
        remaining = count
        for group in order:
            if not remaining:
                break
            start, stop = groups[group]
            available = np.flatnonzero(graph.valid[start:stop]) + start
            take = min(remaining, len(available))
            if take == stop-start:
                mask[start:stop] = True
            elif take:
                ids = available[np.argsort(-physical[available], kind="stable")[:take]]
                mask[ids] = True
            remaining -= take
        assert not remaining and int(mask.sum()) == count, "Compact precision budget is not exact"
        candidates.append(mask)
    return candidates


def allocate_compact_terms(graph: WeightGraph, fraction: float, cfg: Config) -> tuple[np.ndarray, np.ndarray, dict]:
    total = min(int(math.floor(fraction*graph.n_weights)), int(graph.valid.sum()))
    s, _, _ = allocate_terms(graph, 0, "CAPQ", cfg)
    mask = np.zeros(graph.n_weights, bool)
    for step in range(cfg.allocation_rounds if total else 0):
        count = int(math.ceil(total*(step+1)/cfg.allocation_rounds))
        best = (float("inf"), s, mask)
        for proposal in compact_mask_proposals(graph, s, count, cfg):
            candidate, _ = optimize_gauge(graph, proposal, cfg, initial=s,
                                           steps=max(2, cfg.gauge_steps//2))
            cost = graph_specs(graph, candidate, proposal, cfg.exponent_bits)[2]
            if cost < best[0]:
                best = cost, candidate, proposal
        _, s, mask = best
    _, _, error = graph_specs(graph, s, mask, cfg.exponent_bits)
    return s, mask, dict(relative_surrogate=error, reserved_terms=int(mask.sum()),
                         compact_block_size=cfg.compact_block_size, numerical_ridge=graph.numerical_ridge)


def allocate_terms(graph: WeightGraph, fraction: float, method: str, cfg: Config) -> tuple[np.ndarray, np.ndarray, dict]:
    if method == "CAPQ-compact":
        return allocate_compact_terms(graph, fraction, cfg)
    total = int(math.floor(fraction * graph.n_weights))
    total = min(total, int(graph.valid.sum()))
    mask = np.zeros(graph.n_weights, bool)
    use_gauge = method != "CAPQ-no-gauge"
    if method == "CLE+local":
        s = graph.cle_scales(cfg.gauge_bound)
    elif use_gauge:
        zero = np.zeros(graph.n_nodes)
        cle = graph.cle_scales(cfg.gauge_bound)
        candidates = [optimize_gauge(graph, mask, cfg, initial=start)[0] for start in (zero, cle)]
        s = min(candidates, key=lambda a: graph_specs(graph, a, mask, cfg.exponent_bits)[2])
    else:
        s = np.zeros(graph.n_nodes)
    rounds = cfg.allocation_rounds if total else 0
    targets = [int(math.ceil(total * (i + 1) / rounds)) for i in range(rounds)]
    accepted_exchanges = 0
    for target_count in targets:
        cycle_method = method in ("CAPQ", "CAPQ-no-sensitivity")
        gains, local_gain = allocation_scores(graph, s, cfg, cycle_method)
        gains[mask | ~graph.valid] = -np.inf
        needed = target_count - int(mask.sum())
        if needed:
            selected = np.argpartition(gains, -needed)[-needed:]
            mask[selected] = True
        if use_gauge and method != "CLE+local":
            s, _ = optimize_gauge(graph, mask, cfg, initial=s, steps=max(2, cfg.gauge_steps // 2))
        # Reallocate after scale changes, keeping the SAME exact number of
        # selected weights. Evaluate both projected and physical gain proposals
        # with the true quantization objective; accept only genuine reductions.
        gains, local_gain = allocation_scores(graph, s, cfg, cycle_method)
        current_cost = graph_specs(graph, s, mask, cfg.exponent_bits)[2]
        proposals = [top_mask(gains, target_count)]
        if cycle_method:
            proposals.append(top_mask(local_gain, target_count))
        best = (current_cost, s, mask)
        for candidate_mask in proposals:
            if np.array_equal(candidate_mask, mask):
                continue
            candidate_s = s
            if use_gauge and method != "CLE+local":
                candidate_s, _ = optimize_gauge(graph, candidate_mask, cfg, initial=s,
                                                steps=max(2, cfg.gauge_steps // 2))
            cost = graph_specs(graph, candidate_s, candidate_mask, cfg.exponent_bits)[2]
            if cost < best[0] - 1e-12:
                best = cost, candidate_s, candidate_mask
        if best[0] < current_cost - 1e-12:
            accepted_exchanges += 1
        _, s, mask = best
    specs, _, error = graph_specs(graph, s, mask, cfg.exponent_bits)
    used = sum(int(np.count_nonzero(v["mask"] & ~np.isin(v["tag"], [0, 4]))) for v in specs.values())
    return s, mask, dict(relative_surrogate=error, reserved_terms=int(mask.sum()),
                         initially_used_nonpot_terms=used, accepted_exchanges=accepted_exchanges,
                         numerical_ridge=graph.numerical_ridge)


def nested_codebook(minimum: int, exponent_bits: int, depth: int) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Nested dyadic lattices. Each additional plane costs one bit per weight.

    A nonzero value is sign * 2**e * (1 + j / 2**depth), with zero
    explicitly represented. The codebooks nest exactly for depths 0..3.
    This codebook is an engineering component, not a claim of new mathematics.
    """
    if not 0 <= depth <= 3:
        raise ValueError("Nested mantissa depth must be between zero and three")
    count = 2**depth
    exponents = np.repeat(np.arange(1, 2**exponent_bits, dtype=np.uint8), count)
    mantissas = np.tile(np.arange(count, dtype=np.uint8), 2**exponent_bits-1)
    values = np.exp2(minimum + exponents.astype(np.int32)-1) * (1+mantissas.astype(np.float64)/count)
    return (np.concatenate([[0.0], values]), np.concatenate([[0], exponents]).astype(np.uint8),
            np.concatenate([[0], mantissas]).astype(np.uint8))


def group_sizes(count: int, block: int) -> np.ndarray:
    if count < 1 or block < 1:
        raise ValueError("A quantized layer and its block size must be positive")
    sizes = np.full((count+block-1)//block, block, np.int32)
    sizes[-1] = count - block*(len(sizes)-1)
    return sizes


def nested_spec(weight: np.ndarray, alpha: np.ndarray, depths: np.ndarray,
                exponent_bits: int, block: int, minimum: int | None = None) -> dict:
    flat = weight.reshape(-1).astype(np.float64)
    sizes = group_sizes(len(flat), block)
    depths = np.asarray(depths, np.uint8)
    if depths.shape != sizes.shape or np.any(depths > 3):
        raise ValueError("Invalid nested group depths")
    expanded = np.repeat(depths, sizes)
    mag = np.abs(flat)
    def quantize(m: int) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
        code, tag, q = np.empty(len(flat), np.uint8), np.empty(len(flat), np.uint8), np.empty(len(flat))
        for depth in range(4):
            ids = expanded == depth
            if not np.any(ids):
                continue
            values, codes, tags = nested_codebook(m, exponent_bits, depth)
            chosen = np.searchsorted((values[:-1]+values[1:])/2, mag[ids], side="left")
            code[ids], tag[ids], q[ids] = codes[chosen], tags[chosen], values[chosen]
        return code, tag, q
    valid = mag > 1e-12
    if minimum is None:
        highs = np.floor(np.quantile(np.log2(mag[valid]), [0.97,0.99,0.999,1])).astype(int) if valid.any() else [-2]
        candidates = sorted({int(h-(2**exponent_bits-2)+d) for h in highs for d in (-1,0,1)})
        best = (float("inf"), candidates[0])
        for m in candidates:
            q = quantize(m)[2]
            ratio = np.divide(q, mag, out=np.zeros_like(q), where=valid)
            cost = float(np.dot(alpha[valid], (ratio[valid]-1)**2))
            if cost < best[0]:
                best = (cost, m)
        minimum = best[1]
    code, tag, q = quantize(minimum)
    value = (np.sign(flat)*q).astype(np.float32).reshape(weight.shape)
    return dict(kind="nested", shape=list(weight.shape), exponent_bits=int(exponent_bits),
                min_exp=int(minimum), block_size=int(block), group_depths=depths.copy(),
                base_code=code, sign=flat < 0, mantissa=tag, value=value)


class NestedLayout:
    def __init__(self, graph: WeightGraph, block: int):
        self.graph, self.block = graph, block
        self.slices, starts, stops, sizes = {}, [], [], []
        count = 0
        for layer in graph.layers:
            sz = group_sizes(layer.stop-layer.start, block)
            self.slices[layer.name] = slice(count, count+len(sz))
            a = layer.start
            for size in sz:
                starts.append(a); stops.append(a+int(size)); sizes.append(int(size)); a += int(size)
            count += len(sz)
        self.starts = np.asarray(starts, np.int64)
        self.stops = np.asarray(stops, np.int64)
        self.sizes = np.asarray(sizes, np.int32)
        self.gid = np.repeat(np.arange(count, dtype=np.int32), self.sizes)
        self.count = count

    def specs(self, s: np.ndarray, depths: np.ndarray, bits: int,
              windows: dict[str, int] | None = None, metadata_only: bool = False) -> dict:
        result = {}
        weight = None if metadata_only else self.graph.weights*np.exp2(self.graph.shift(s))
        for layer in self.graph.layers:
            d = depths[self.slices[layer.name]]
            minimum = None if windows is None else windows[layer.name]
            if metadata_only:
                if minimum is None:
                    raise ValueError("Exact byte accounting requires fixed exponent windows")
                result[layer.name] = dict(kind="nested", shape=list(layer.shape), exponent_bits=bits,
                                         min_exp=minimum, block_size=self.block, group_depths=d)
            else:
                a = slice(layer.start, layer.stop)
                result[layer.name] = nested_spec(weight[a].reshape(layer.shape), self.graph.alpha[a], d,
                                                bits, self.block, minimum)
        return result


class TaskMetric:
    """PSD diagonal-plus-sketch metric, shared by gauge refinement and allocation."""
    def __init__(self,graph: WeightGraph,mix: float):
        self.graph,self.diagonal_weight=graph,1-mix
        self.V=None
        self.VB=None
        self.L=graph.L
        self.factor=graph.factor
        if mix:
            raw=np.asarray(graph.task_sketch,np.float64)
            norm=np.sqrt(np.sum(raw**2,axis=0))
            normalized=np.divide(raw,norm[None,:],out=np.zeros_like(raw),where=norm[None,:]>1e-30)
            self.V=normalized*np.sqrt(mix*graph.alpha)[None,:]
            self.VB=np.stack([graph.rhs(row,weights=np.ones(graph.n_weights)) for row in self.V])
            self.L=self.diagonal_weight*graph.L+self.VB.T@self.VB
            self.factor=sla.cho_factor(self.L,check_finite=False)

    def dual(self,v: np.ndarray) -> np.ndarray:
        result=self.diagonal_weight*self.graph.alpha*v
        if self.V is not None:
            result+=self.V.T@(self.V@v)
        return result

    def energy(self,v: np.ndarray) -> float:
        return float(np.dot(v,self.dual(v)))

    def solve_scales(self,v: np.ndarray,bound: float,warm: np.ndarray) -> np.ndarray:
        b=self.graph.rhs(self.dual(v),weights=np.ones(self.graph.n_weights))
        solution=-sla.cho_solve(self.factor,b,check_finite=False)
        if np.max(np.abs(solution),initial=0)<=bound:
            return solution
        def fun(s: np.ndarray) -> tuple[float,np.ndarray]:
            return float(0.5*s@(self.L@s)+b@s),self.L@s+b
        result=sopt.minimize(fun,np.clip(warm,-bound,bound),jac=True,method="L-BFGS-B",
                             bounds=[(-bound,bound)]*self.graph.n_nodes,
                             options=dict(maxiter=150,ftol=1e-12,gtol=1e-7))
        if not np.isfinite(result.x).all():
            raise FloatingPointError("Invalid coupled bounded gauge solve")
        return result.x

    def gauss_newton(self,ratio: np.ndarray) -> np.ndarray:
        graph=self.graph
        H=graph.laplacian(self.diagonal_weight*graph.alpha*ratio**2)
        if self.V is not None:
            scaled=self.V*ratio[None,:]
            JB=np.stack([graph.rhs(row,weights=np.ones(graph.n_weights)) for row in scaled])
            H+=JB.T@JB
        H.flat[::graph.n_nodes+1]+=max(float(np.max(np.diag(H),initial=0)),1)*1e-9
        rhs=graph.rhs(ratio*self.dual(ratio-1),weights=np.ones(graph.n_weights))/math.log(2)
        return sla.solve(H,rhs,assume_a="pos",check_finite=False)


def nested_objective(graph: WeightGraph, layout: NestedLayout, s: np.ndarray, depths: np.ndarray,
                     cfg: Config, windows: dict[str, int] | None = None) -> tuple[dict, np.ndarray, float]:
    specs = layout.specs(s, depths, cfg.exponent_bits, windows)
    q = np.concatenate([np.abs(specs[layer.name]["value"].reshape(-1)).astype(np.float64) for layer in graph.layers])
    mag = graph.magnitude*np.exp2(graph.shift(s))
    ratio = np.divide(q, mag, out=np.zeros_like(q), where=graph.valid)
    metric=getattr(graph,"task_metric",None)
    cost=(metric.energy(ratio-1) if metric is not None else float(np.dot(graph.alpha,(ratio-1)**2)))
    cost/=max(float(graph.alpha.sum()),1e-30)
    target = np.empty_like(q)
    for layer in graph.layers:
        a = slice(layer.start, layer.stop)
        target[a] = np.log2(np.where(q[a]>0, q[a], 2.0**specs[layer.name]["min_exp"]))
    return specs, target, cost


def optimize_nested_gauge(graph: WeightGraph, layout: NestedLayout, depths: np.ndarray, cfg: Config,
                          initial: np.ndarray, steps: int | None = None) -> tuple[np.ndarray, dict, float]:
    s = initial.copy()
    specs, target, cost = nested_objective(graph, layout, s, depths, cfg)
    for _ in range(cfg.gauge_steps if steps is None else steps):
        windows = {name: sp["min_exp"] for name, sp in specs.items()}
        metric=getattr(graph,"task_metric",None)
        proposal=(metric.solve_scales(graph.x-target,cfg.gauge_bound,s) if metric is not None else
                  graph.solve_scales(target,cfg.gauge_bound,s))
        q = np.concatenate([np.abs(specs[layer.name]["value"].reshape(-1)) for layer in graph.layers])
        mag = graph.magnitude*np.exp2(graph.shift(s))
        ratio = np.divide(q, mag, out=np.zeros_like(mag), where=graph.valid)
        if metric is not None:
            delta=metric.gauss_newton(ratio)
        else:
            H=graph.laplacian(graph.alpha*ratio**2)
            H.flat[::graph.n_nodes+1]+=max(float(np.max(np.diag(H),initial=0)),1)*1e-9
            delta=sla.solve(H,graph.rhs(ratio*(ratio-1))/math.log(2),assume_a="pos",check_finite=False)
        directions = (proposal, np.clip(s+delta, -cfg.gauge_bound, cfg.gauge_bound))
        best = (cost, s, specs, target)
        for direction in directions:
            for fraction in (1.0, 0.5, 0.25):
                candidate = s+fraction*(direction-s)
                sp, t, c = nested_objective(graph, layout, candidate, depths, cfg, windows)
                if c < best[0]-1e-12:
                    best = c, candidate, sp, t
        if best[0] >= cost-1e-12:
            break
        cost, s, specs, target = best
        sp, t, c = nested_objective(graph, layout, s, depths, cfg)
        if c <= cost:
            specs, target, cost = sp, t, c
    return s, specs, cost


class ConditionalGeometry:
    """Fixed-code log-quadratic marginals, with interactions across blocks.

    For H=(1-eta)A+V^T V, Q=H-H B (B^T H B)^(-1) B^T H,
    and residual v=x-q, an upgrade mu has gain 2 v^T Q mu-mu^T Q mu.
    For disjoint blocks g,h, choosing g changes h's gain by
    2*(j_h^T L^(-1) j_g-z_h^T z_g), where j=B^T H mu, z=V mu.
    We compute this exactly within each shortlist, and rebuild the global
    conditional scores after each batch. No global-optimality claim is made.
    """
    def __init__(self, graph: WeightGraph, layout: NestedLayout, s: np.ndarray,
                 windows: dict[str,int], cfg: Config, project_gauge: bool = True,
                 sketch_mix: float = 0.0):
        self.graph, self.layout = graph, layout
        self.project_gauge=project_gauge
        self.diagonal_weight=1-sketch_mix
        self.V=None
        self.VB=None
        self.factor=graph.factor
        raw=getattr(graph,"task_sketch",None)
        if sketch_mix and raw is not None:
            if not 0<sketch_mix<1:
                raise ValueError("Sketch shrinkage must lie strictly between zero and one")
            raw=np.asarray(raw,np.float64)
            norm=np.sqrt(np.sum(raw**2,axis=0))
            normalized=np.divide(raw,norm[None,:],out=np.zeros_like(raw),where=norm[None,:]>1e-30)
            self.V=normalized*np.sqrt(sketch_mix*graph.alpha)[None,:]
            self.VB=np.stack([graph.rhs(row,weights=np.ones(graph.n_weights)) for row in self.V])
            if project_gauge:
                L=self.diagonal_weight*graph.L+self.VB.T@self.VB
                self.factor=sla.cho_factor(L,check_finite=False)
        elif sketch_mix:
            raise ValueError("A nonzero sketch mixture requires real calibration Jacobian probes")
        self.target = np.empty((cfg.rd_max_depth+1, graph.n_weights), np.float64)
        self.physical = np.empty_like(self.target)
        mag = graph.magnitude*np.exp2(graph.shift(s))
        for depth in range(cfg.rd_max_depth+1):
            for layer in graph.layers:
                a = slice(layer.start, layer.stop)
                values, _, _ = nested_codebook(windows[layer.name], cfg.exponent_bits, depth)
                chosen = np.searchsorted((values[:-1]+values[1:])/2, mag[a], side="left")
                q = values[chosen]
                self.target[depth,a] = np.log2(np.where(q>0, q, 2.0**windows[layer.name]))
                ratio = np.divide(q, mag[a], out=np.zeros_like(q), where=graph.valid[a])
                self.physical[depth,a] = graph.alpha[a]*(ratio-1)**2
        self.mu = np.diff(self.target, axis=0)
        self.J, self.T, self.Z, self.curvature, self.local = [], [], [], [], []
        h, t = graph.head >= 0, graph.tail >= 0
        for depth in range(cfg.rd_max_depth):
            mu = self.mu[depth]
            J = np.zeros((graph.n_nodes if project_gauge else 1, layout.count), np.float64)
            if project_gauge:
                np.add.at(J, (graph.head[h], layout.gid[h]), self.diagonal_weight*graph.alpha[h]*mu[h])
                np.add.at(J, (graph.tail[t], layout.gid[t]), -self.diagonal_weight*graph.alpha[t]*mu[t])
            Z=None
            curvature=self.diagonal_weight*np.bincount(layout.gid,weights=graph.alpha*mu**2,minlength=layout.count)
            if self.V is not None:
                Z=np.stack([np.bincount(layout.gid,weights=row*mu,minlength=layout.count) for row in self.V])
                curvature+=np.sum(Z**2,axis=0)
                if project_gauge:
                    J+=self.VB.T@Z
            T=sla.cho_solve(self.factor,J,check_finite=False) if project_gauge else np.zeros_like(J)
            curvature-=np.sum(J*T,axis=0)
            local = np.bincount(layout.gid, weights=self.physical[depth]-self.physical[depth+1], minlength=layout.count)
            self.J.append(J); self.T.append(T);self.Z.append(Z)
            self.curvature.append(curvature); self.local.append(local)
        self.curvature, self.local = np.asarray(self.curvature), np.asarray(self.local)
        self.maximum = cfg.rd_max_depth

    def dual(self,v: np.ndarray) -> np.ndarray:
        result=self.diagonal_weight*self.graph.alpha*v
        if self.V is not None:
            result+=self.V.T@(self.V@v)
        return result

    def project(self,v: np.ndarray) -> np.ndarray:
        if not self.project_gauge:
            return v
        rhs=self.diagonal_weight*self.graph.rhs(v)
        if self.V is not None:
            rhs+=self.VB.T@(self.V@v)
        s=-sla.cho_solve(self.factor,rhs,check_finite=False)
        return v+self.graph.shift(s)

    def energy(self,v: np.ndarray) -> float:
        residual=self.project(v)
        return float(np.dot(residual,self.dual(residual)))

    def scores(self, depths: np.ndarray, conditional: bool = True) -> np.ndarray:
        graph, layout = self.graph, self.layout
        active = np.minimum(depths, self.maximum-1)
        if not conditional:
            score = self.local[active,np.arange(layout.count)].copy()
        else:
            wdepth = depths[layout.gid]
            target = self.target[wdepth,np.arange(graph.n_weights)]
            residual = self.project(graph.x-target)
            mu = self.mu[active[layout.gid],np.arange(graph.n_weights)]
            score = 2*np.bincount(layout.gid, weights=self.dual(residual)*mu, minlength=layout.count)
            score -= self.curvature[active,np.arange(layout.count)]
        score[depths >= self.maximum] = -np.inf
        return score

    def interaction(self, ids: np.ndarray, depths: np.ndarray) -> np.ndarray:
        columns_j = np.column_stack([self.J[int(depths[g])][:,g] for g in ids])
        columns_t = np.column_stack([self.T[int(depths[g])][:,g] for g in ids])
        result=columns_j.T@columns_t
        if self.V is not None:
            z=np.column_stack([self.Z[int(depths[g])][:,g] for g in ids])
            result-=z.T@z
        return result


def verify_nested_method(graph: WeightGraph, cfg: Config, seed: int) -> dict:
    layout=NestedLayout(graph,cfg.compact_block_size)
    rng=np.random.default_rng(seed+60601)
    depths=rng.integers(0,4,layout.count,dtype=np.uint8)
    scales=np.zeros(graph.n_nodes)
    specs=layout.specs(scales,depths,cfg.exponent_bits)
    windows={name:sp["min_exp"] for name,sp in specs.items()}
    student=install_quantizers(graph.model,specs)
    materialized,exported=materialize_student(student,graph.arch,[layer.name for layer in graph.layers])
    blob=encode_model(materialized,graph.arch,exported)
    assert_roundtrip(materialized,blob)
    assert len(blob)==int(encode_model(materialized,graph.arch,exported,size_only=True)), "Byte estimator differs from file"
    for minimum in (-12,-4,1):
        previous=None
        for depth in range(4):
            values,_,_=nested_codebook(minimum,cfg.exponent_bits,depth)
            if previous is not None:
                assert np.isin(previous,values).all(), "Mantissa lattices must be nested"
            previous=values
    geometry=ConditionalGeometry(graph,layout,scales,windows,cfg)
    d=np.zeros(layout.count,np.uint8)
    v=graph.x-geometry.target[0]
    before=float(np.dot(graph.alpha,graph.project_residual(v)**2))
    ids=rng.choice(layout.count,size=min(2,layout.count),replace=False)
    mu=np.zeros(graph.n_weights)
    for group in ids:
        a=slice(layout.starts[group],layout.stops[group])
        mu[a]=geometry.mu[0,a]
    scores=geometry.scores(d,True)
    gram=geometry.interaction(ids,d)
    predicted_gain=float(scores[ids].sum()+2*np.triu(gram,1).sum())
    after=float(np.dot(graph.alpha,graph.project_residual(v-mu)**2))
    assert abs(before-after-predicted_gain)<2e-7*max(before,1), "Incorrect joint conditional marginal"
    if len(ids)==2:
        d[ids[0]]=1
        updated=geometry.scores(d,True)[ids[1]]
        assert abs(updated-(scores[ids[1]]+2*gram[1,0]))<2e-7*max(abs(updated),abs(scores[ids[1]]),1), "Conditional score update is not exact"
    del geometry,student
    return dict(nested_lattices_verified=True,nested_codec_verified=True,
                exact_byte_estimator_verified=True,conditional_interaction_verified=True)


def exact_nested_bytes(graph: WeightGraph, layout: NestedLayout, depths: np.ndarray,
                       windows: dict[str,int], cfg: Config) -> int:
    specs = layout.specs(np.zeros(graph.n_nodes), depths, cfg.exponent_bits, windows, metadata_only=True)
    return int(encode_model(graph.model, graph.arch, specs, size_only=True))


def greedy_nested_depths(geometry: ConditionalGeometry, windows: dict[str,int], cap: int,
                         method: str, cfg: Config) -> tuple[np.ndarray, dict]:
    graph, layout = geometry.graph, geometry.layout
    depths = np.zeros(layout.count, np.uint8)
    size = exact_nested_bytes(graph, layout, depths, windows, cfg)
    if size > cap:
        raise ValueError(f"Byte budget {cap} is below the {size}-byte base representation")
    rejected = np.zeros(layout.count, bool)
    batches = upgrades = within_batch_updates = 0
    static_scores = None
    while True:
        use_cycle = method != "Local-RD"
        if method == "Static-RD" and static_scores is not None:
            score = static_scores[np.minimum(depths,geometry.maximum-1), np.arange(layout.count)].copy()
            score[depths>=geometry.maximum] = -np.inf
        else:
            score = geometry.scores(depths, use_cycle)
        # Upgrade cost in payload bits is group length. Exact metadata and
        # padding costs are measured for EVERY proposed acceptance below.
        rank = score / np.maximum(layout.sizes/8, 1)
        rank[rejected] = -np.inf
        eligible = np.flatnonzero(np.isfinite(rank) & (score > 1e-12))
        if not len(eligible):
            break
        keep = min(cfg.rd_shortlist,len(eligible))
        pool = eligible[np.argsort(-rank[eligible],kind="stable")[:keep]]
        gains = score[pool].copy()
        gram = geometry.interaction(pool, depths) if method in ("CAPQ-RD","Diagonal-RD","RD-no-gauge") else None
        accepted = 0
        for _ in range(min(cfg.rd_batch,len(pool))):
            ratios = gains/np.maximum(layout.sizes[pool]/8,1)
            pick = int(np.argmax(ratios))
            if not np.isfinite(ratios[pick]) or gains[pick] <= 1e-12:
                break
            group = int(pool[pick])
            depths[group] += 1
            proposed = exact_nested_bytes(graph, layout, depths, windows, cfg)
            if proposed > cap:
                depths[group] -= 1
                rejected[group] = True
            else:
                size = proposed; accepted += 1; upgrades += 1
                if gram is not None:
                    gains += 2*gram[:,pick]
                    within_batch_updates += 1
            gains[pick] = -np.inf
        batches += 1
        if method == "Static-RD" and static_scores is None:
            # Static control: score each possible depth against its uniform
            # lower-depth code vector, independent of the selected other blocks.
            static_scores = np.empty((geometry.maximum,layout.count))
            for k in range(geometry.maximum):
                static_scores[k] = geometry.scores(np.full(layout.count,k,np.uint8),True)
        if not accepted:
            break
        # Previously rejected groups can become affordable after a different
        # depth-map encoding, but each has at most one retry per accepted batch.
        if accepted:
            rejected[:] = False
    assert exact_nested_bytes(graph,layout,depths,windows,cfg) == size <= cap
    return depths, dict(allocation_batches=batches, accepted_bitplane_upgrades=upgrades,
                        conditional_within_batch_updates=within_batch_updates)


def conditional_depth_swaps(geometry: ConditionalGeometry, depths: np.ndarray, windows: dict[str,int],
                             cap: int, cfg: Config) -> tuple[np.ndarray,int]:
    """Exchange an upgrade and a downgrade using their JOINT projected gain.

    The gain is exact for the stated fixed-code, unconstrained quadratic.
    Feasibility uses actual codec bytes, including nonadditive depth-map cost.
    We subsequently safeguard the whole proposal using finite-lattice error.
    """
    graph,layout=geometry.graph,geometry.layout
    depths=depths.copy()
    accepted=0
    for _ in range(cfg.rd_swaps):
        up_scores=geometry.scores(depths,True)
        wdepth=depths[layout.gid]
        residual=geometry.project(graph.x-geometry.target[wdepth,np.arange(graph.n_weights)])
        lower=np.maximum(depths.astype(np.int32)-1,0)
        down_mu=-geometry.mu[lower[layout.gid],np.arange(graph.n_weights)]
        down_scores=2*np.bincount(layout.gid,weights=geometry.dual(residual)*down_mu,minlength=layout.count)
        down_scores-=geometry.curvature[lower,np.arange(layout.count)]
        down_scores[depths==0]=-np.inf
        ups=np.flatnonzero(np.isfinite(up_scores)&(up_scores>1e-12))
        downs=np.flatnonzero(np.isfinite(down_scores))
        if not len(ups) or not len(downs):
            break
        ups=ups[np.argsort(-up_scores[ups]/layout.sizes[ups],kind="stable")[:48]]
        downs=downs[np.argsort(-down_scores[downs]/layout.sizes[downs],kind="stable")[:48]]
        Ju=np.column_stack([geometry.J[int(depths[g])][:,g] for g in ups])
        Td=np.column_stack([-geometry.T[int(depths[g])-1][:,g] for g in downs])
        gains=up_scores[ups,None]+down_scores[None,downs]+2*(Ju.T@Td)
        if geometry.V is not None:
            Zu=np.column_stack([geometry.Z[int(depths[g])][:,g] for g in ups])
            Zd=np.column_stack([-geometry.Z[int(depths[g])-1][:,g] for g in downs])
            gains-=2*(Zu.T@Zd)
        gains[ups[:,None]==downs[None,:]]=-np.inf
        changed=False
        for index in np.argsort(-gains.reshape(-1),kind="stable")[:96]:
            i,j=np.unravel_index(index,gains.shape)
            if gains[i,j]<=1e-12:
                break
            up,down=int(ups[i]),int(downs[j])
            depths[up]+=1;depths[down]-=1
            if exact_nested_bytes(graph,layout,depths,windows,cfg)<=cap:
                accepted+=1;changed=True;break
            depths[up]-=1;depths[down]+=1
        if not changed:
            break
    return depths,accepted


def allocate_rate_distortion(graph: WeightGraph, reference_bits: int, method: str, cfg: Config,
                             initial: tuple | None = None) -> tuple[np.ndarray,dict,dict]:
    gauge_mix=0.0 if method=="Diagonal-RD" else cfg.rd_sketch_mix
    graph.task_metric=TaskMetric(graph,gauge_mix)
    layout = NestedLayout(graph,cfg.compact_block_size)
    uniform = {layer.name:uniform_spec(module_at(graph.model,layer.name).weight.detach().cpu().numpy(),reference_bits)
               for layer in graph.layers}
    cap = int(encode_model(graph.model,graph.arch,uniform,size_only=True))
    zero_depths = np.zeros(layout.count,np.uint8)
    if method == "RD-no-gauge":
        scales = np.zeros(graph.n_nodes)
        specs, _, cost = nested_objective(graph,layout,scales,zero_depths,cfg)
    elif initial is not None:
        scales,specs,cost=initial
        scales=scales.copy();specs=copy.deepcopy(specs)
    else:
        starts = (np.zeros(graph.n_nodes),graph.cle_scales(cfg.gauge_bound))
        candidates = [optimize_nested_gauge(graph,layout,zero_depths,cfg,start) for start in starts]
        scales,specs,cost = min(candidates,key=lambda c:c[2])
    best = None
    for iteration in range(cfg.rd_passes):
        windows = {name:sp["min_exp"] for name,sp in specs.items()}
        mix=cfg.rd_sketch_mix if method in ("CAPQ-RD","Static-RD","RD-no-gauge") else 0.0
        geometry = ConditionalGeometry(graph,layout,scales,windows,cfg,project_gauge=method!="RD-no-gauge",sketch_mix=mix)
        depths, diagnostics = greedy_nested_depths(geometry,windows,cap,method,cfg)
        diagnostics["accepted_conditional_swaps"]=0
        if method in ("CAPQ-RD","Diagonal-RD") and cfg.rd_swaps:
            exchanged,swaps=conditional_depth_swaps(geometry,depths,windows,cap,cfg)
            original_cost=nested_objective(graph,layout,scales,depths,cfg,windows)[2]
            exchange_cost=nested_objective(graph,layout,scales,exchanged,cfg,windows)[2]
            if exchange_cost<original_cost-1e-12:
                depths=exchanged
                diagnostics["accepted_conditional_swaps"]=swaps
        del geometry
        if method != "RD-no-gauge":
            proposal,new_specs,new_cost = optimize_nested_gauge(graph,layout,depths,cfg,scales,
                                                               steps=max(2,cfg.gauge_steps//2))
        else:
            proposal = scales.copy()
            new_specs,_,new_cost = nested_objective(graph,layout,proposal,depths,cfg,windows)
        # Exponent-window metadata can change length by a few bytes. Repair
        # with a feasible retained window instead of silently overshooting.
        new_windows = {name:sp["min_exp"] for name,sp in new_specs.items()}
        size = exact_nested_bytes(graph,layout,depths,new_windows,cfg)
        if size > cap:
            new_specs,_,new_cost = nested_objective(graph,layout,proposal,depths,cfg,windows)
            new_windows=windows
            size=exact_nested_bytes(graph,layout,depths,windows,cfg)
        assert size <= cap, "A repaired allocation must satisfy the packed-byte cap"
        item=(new_cost,proposal,new_specs,depths.copy(),diagnostics,size)
        if best is None or new_cost < best[0]:
            best=item
        scales,specs = proposal,new_specs
    cost,scales,specs,depths,diagnostics,size = best
    info=dict(relative_surrogate=cost,byte_cap=cap,allocation_bytes=size,
              byte_cap_verified=True,reference_uniform_bits=reference_bits,
              mean_mantissa_bits=float(np.dot(depths,layout.sizes)/graph.n_weights),
              depth0_fraction=float(layout.sizes[depths==0].sum()/graph.n_weights),
              depth1_fraction=float(layout.sizes[depths==1].sum()/graph.n_weights),
              depth2_fraction=float(layout.sizes[depths==2].sum()/graph.n_weights),
              depth3_fraction=float(layout.sizes[depths==3].sum()/graph.n_weights),
              compact_block_size=cfg.compact_block_size, numerical_ridge=graph.numerical_ridge,
              sketch_mix=cfg.rd_sketch_mix if method in ("CAPQ-RD","Static-RD","RD-no-gauge") else 0.0,
              gauge_sketch_mix=gauge_mix,
              **diagnostics)
    return scales,specs,info


@torch.no_grad()
def calibration_kl(graph: WeightGraph, scales: np.ndarray, specs: dict, x: torch.Tensor,
                   cfg: Config, device: torch.device) -> float:
    student=install_quantizers(graph.apply(scales),specs).to(device).eval()
    teacher=graph.model.to(device).eval()
    total=0.0
    for batch in x.split(cfg.batch_size):
        batch=batch.to(device)
        target=teacher(batch)
        logits=student(batch)
        temperature=cfg.kd_temperature
        total+=float(F.kl_div(F.log_softmax(logits/temperature,dim=1),
                            F.softmax(target/temperature,dim=1),reduction="sum"))*temperature**2
    del student
    return total/max(len(x),1)


def prepare_gauge_initials(graph: WeightGraph,cfg: Config,x: torch.Tensor,device: torch.device) -> dict:
    """Select a metric on calibration data BEFORE allocating any byte budget.

    The same metric and initialization are shared by local/coupled/static
    controls. The diagonal ablation fixes eta=0. The choice {0, configured eta}
    is fixed in advance; no validation/test metric is used in this choice.
    """
    started=time.monotonic()
    layout=NestedLayout(graph,cfg.compact_block_size)
    depths=np.zeros(layout.count,np.uint8)
    candidates={}
    losses={}
    for mix in (0.0,cfg.rd_sketch_mix):
        graph.task_metric=TaskMetric(graph,mix)
        starts=(np.zeros(graph.n_nodes),graph.cle_scales(cfg.gauge_bound))
        items=[optimize_nested_gauge(graph,layout,depths,cfg,start) for start in starts]
        item=min(items,key=lambda a:a[2])
        candidates[mix]=item
        losses[mix]=calibration_kl(graph,item[0],item[1],x,cfg,device)
    selected=min(losses,key=losses.get) if cfg.rd_adaptive_metric else cfg.rd_sketch_mix
    log(f"Shared calibration metric: eta=0 KL={losses[0.0]:.6g}, eta={cfg.rd_sketch_mix:g} KL={losses[cfg.rd_sketch_mix]:.6g}; selected eta={selected:g}")
    return dict(initials=candidates,selected_mix=selected,calibration_losses=losses,
                shared_gauge_seconds=time.monotonic()-started)


def allocate_guarded_rd(graph: WeightGraph,bits: int,method: str,cfg: Config,x: torch.Tensor,
                        device: torch.device,cache: dict) -> tuple[np.ndarray,dict,dict]:
    """A real-data safeguard, shared by the coupled variants and ablations.

    Candidate selection uses TRAINING calibration examples and teacher KL,
    never test labels/metrics. All selected students receive the same KD steps.
    This guarantees only calibration KL no worse than the local proposal,
    not an accuracy guarantee. Extra allocation cost is included in runtime.
    """
    gauges_were_reused="_gauges" in cache
    if "_gauges" not in cache:
        cache["_gauges"]=prepare_gauge_initials(graph,cfg,x,device)
    gauges=cache["_gauges"]
    mix=0.0 if method=="Diagonal-RD" else gauges["selected_mix"]
    active_cfg=dataclasses.replace(cfg,rd_sketch_mix=mix)
    initial=None if method=="RD-no-gauge" else gauges["initials"][mix]
    key=(bits,method=="RD-no-gauge",method=="Diagonal-RD")
    def metadata(item: tuple) -> tuple:
        item[2].update(selected_metric_mix=mix,shared_gauge_selection_seconds=gauges["shared_gauge_seconds"],
                       reused_gauge_selection_seconds=gauges["shared_gauge_seconds"] if gauges_were_reused else 0.0,
                       diagonal_base_calibration_kl=gauges["calibration_losses"][0.0],
                       sketch_base_calibration_kl=gauges["calibration_losses"][cfg.rd_sketch_mix])
        return item
    if method=="Local-RD":
        started=time.monotonic()
        item=metadata(allocate_rate_distortion(graph,bits,method,active_cfg,initial))
        item[2]["shared_local_allocation_seconds"]=time.monotonic()-started
        cache[key]=item
        return item
    proposal=metadata(allocate_rate_distortion(graph,bits,method,active_cfg,initial))
    if not cfg.rd_calibration_guard:
        return proposal
    if key not in cache:
        started=time.monotonic()
        if method=="RD-no-gauge":
            local_cfg=dataclasses.replace(active_cfg,gauge_bound=0.0)
        elif method=="Diagonal-RD":
            local_cfg=dataclasses.replace(active_cfg,rd_sketch_mix=0.0)
        else:
            local_cfg=active_cfg
        cache[key]=metadata(allocate_rate_distortion(graph,bits,"Local-RD",local_cfg,initial))
        cache[key][2]["shared_local_allocation_seconds"]=time.monotonic()-started
        cache_was_reused=False
    else:
        cache_was_reused=True
    local=cache[key]
    proposal_kl=calibration_kl(graph,proposal[0],proposal[1],x,cfg,device)
    local_kl=calibration_kl(graph,local[0],local[1],x,cfg,device)
    selected=local if local_kl<proposal_kl else proposal
    info=dict(selected[2],guard_selected="local" if selected is local else "coupled",
          coupled_calibration_kl=proposal_kl,local_calibration_kl=local_kl,
          selected_calibration_kl=min(local_kl,proposal_kl),calibration_guard_verified=True,
          requested_sketch_mix=cfg.rd_sketch_mix if method!="Diagonal-RD" else 0.0,
          shared_local_allocation_seconds=local[2].get("shared_local_allocation_seconds",0.0),
          reused_guard_allocation_seconds=local[2].get("shared_local_allocation_seconds",0.0) if cache_was_reused else 0.0)
    log(f"Calibration guard {method}: coupled KL={proposal_kl:.6g}, local KL={local_kl:.6g}; selected={info['guard_selected']}")
    return selected[0],selected[1],info


def verify_sketch_geometry(graph: WeightGraph,cfg: Config,seed: int) -> dict:
    layout=NestedLayout(graph,cfg.compact_block_size)
    d=np.zeros(layout.count,np.uint8)
    s=np.zeros(graph.n_nodes)
    specs=layout.specs(s,d,cfg.exponent_bits)
    windows={name:sp["min_exp"] for name,sp in specs.items()}
    geometry=ConditionalGeometry(graph,layout,s,windows,cfg,sketch_mix=cfg.rd_sketch_mix)
    rng=np.random.default_rng(seed+70701)
    ids=rng.choice(layout.count,size=min(2,layout.count),replace=False)
    v=graph.x-geometry.target[0]
    mu=np.zeros_like(v)
    for group in ids:
        a=slice(layout.starts[group],layout.stops[group])
        mu[a]=geometry.mu[0,a]
    score=geometry.scores(d,True)
    interactions=geometry.interaction(ids,d)
    gain=float(score[ids].sum()+2*np.triu(interactions,1).sum())
    before=geometry.energy(v)
    after=geometry.energy(v-mu)
    assert abs(before-after-gain)<3e-7*max(before,1),"Incorrect low-rank quotient marginal"
    normal=graph.rhs(geometry.dual(geometry.project(v)),weights=np.ones(graph.n_weights))
    assert np.linalg.norm(normal)<3e-6*max(np.linalg.norm(graph.rhs(geometry.dual(v),weights=np.ones(graph.n_weights))),1),"Quotient residual is not H-orthogonal to the gauge subspace"
    if len(ids)==2:
        d[ids[0]]=1
        new=geometry.scores(d,True)[ids[1]]
        assert abs(new-score[ids[1]]-2*interactions[1,0])<3e-7*max(abs(new),abs(score[ids[1]]),1),"Incorrect task-coupled conditional update"
    metric=TaskMetric(graph,cfg.rd_sketch_mix)
    q=np.concatenate([np.abs(specs[layer.name]["value"].reshape(-1)) for layer in graph.layers])
    ratio=np.divide(q,graph.magnitude,out=np.zeros_like(graph.magnitude),where=graph.valid)
    direction=rng.normal(size=graph.n_nodes)
    gradient=-math.log(2)*graph.rhs(ratio*metric.dual(ratio-1),weights=np.ones(graph.n_weights))
    eps=1e-5
    high=ratio*np.exp2(-eps*graph.shift(direction))-1
    low=ratio*np.exp2(eps*graph.shift(direction))-1
    numeric=(0.5*metric.energy(high)-0.5*metric.energy(low))/(2*eps)
    analytic=float(gradient@direction)
    assert abs(numeric-analytic)<3e-5*max(abs(numeric),abs(analytic),1),"Incorrect coupled gauge gradient"
    del geometry
    return dict(low_rank_quotient_verified=True,task_coupled_update_verified=True,
                gauge_orthogonality_verified=True,coupled_gauge_gradient_verified=True,
                sketch_rank=int(graph.task_sketch.shape[0]))


def make_loader(dataset: Dataset, cfg: Config, shuffle: bool = False, seed: int = 0,
                batch_size: int | None = None) -> DataLoader:
    generator = torch.Generator().manual_seed(seed)
    return DataLoader(dataset, batch_size=batch_size or cfg.batch_size, shuffle=shuffle,
                      num_workers=cfg.workers, pin_memory=torch.cuda.is_available(),
                      generator=generator, persistent_workers=False)


@torch.no_grad()
def evaluate(model: nn.Module, loader: DataLoader, device: torch.device,
             reference: nn.Module | None = None, temperature: float = 2.0) -> dict:
    model.eval()
    n = correct = 0
    loss = 0.0
    teacher_kl=error_energy=reference_energy=0.0
    if reference is not None:
        reference.eval()
    for x, y in loader:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        logits = model(x)
        if not torch.isfinite(logits).all():
            raise FloatingPointError("Non-finite logits during evaluation")
        correct += int((logits.argmax(1) == y).sum())
        loss += float(F.cross_entropy(logits, y, reduction="sum"))
        if reference is not None:
            target=logits if reference is model else reference(x)
            logp=F.log_softmax(target.double()/temperature,dim=1)
            logq=F.log_softmax(logits.double()/temperature,dim=1)
            teacher_kl+=float((logp.exp()*(logp-logq)).sum())*temperature**2
            centered_target=target.double()-target.double().mean(dim=1,keepdim=True)
            centered_student=logits.double()-logits.double().mean(dim=1,keepdim=True)
            error_energy+=float(((centered_student-centered_target)**2).sum())
            reference_energy+=float((centered_target**2).sum())
        n += len(y)
    if n == 0:
        raise ValueError("Empty evaluation dataset")
    result=dict(accuracy=100*correct/n,loss=loss/n,n=n)
    if reference is not None:
        result.update(teacher_kl=max(teacher_kl/n,0.0),logit_nmse=error_energy/max(reference_energy,1e-30))
    return result


def cpu_state(model: nn.Module) -> dict:
    return {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}


def train_teacher(data: DataBundle, dataset: str, seed: int, cfg: Config,
                  run_dir: Path, device: torch.device) -> tuple[nn.Module, dict]:
    epochs = {"cifar100": cfg.cifar_epochs, "speechcommands": cfg.speech_epochs,
              "fashionmnist": cfg.fashion_epochs, "digits": cfg.digits_epochs}[dataset]
    path = run_dir / "teachers" / f"{dataset}_seed{seed}.pt"
    seed_everything(seed)
    model = build_model(data.architecture).to(device)
    if cfg.teacher_cache:
        cache = Path(cfg.teacher_cache)
        protocol = json.loads((cache/"protocol.json").read_text())
        previous = protocol["configuration"]
        epoch_key = {"cifar100":"cifar_epochs", "speechcommands":"speech_epochs",
                     "fashionmnist":"fashion_epochs", "digits":"digits_epochs"}[dataset]
        if (dataset not in previous["datasets"] or protocol["split_seed"] != SPLIT_SEED or
            tuple(protocol["seeds"]) != SEEDS or previous["smoke"] != cfg.smoke or
            previous[epoch_key] != epochs or previous["batch_size"] != cfg.batch_size or
            previous["workers"] != cfg.workers or protocol["torch"] != torch.__version__):
            raise ValueError("Selected teacher cache has a different training/data protocol")
        old_data = json.loads((cache/f"dataset_{dataset}.json").read_text())
        if old_data != json.loads(json.dumps(data.dataset_info)):
            raise ValueError("Selected teacher cache has different dataset splits/preprocessing")
        checkpoint = local_torch_load(cache/"teachers"/f"{dataset}_seed{seed}.pt", device)
        if not checkpoint.get("complete", False):
            raise ValueError("Only completed teachers can be explicitly reused")
        model.load_state_dict(checkpoint["best_state"], strict=True)
        log(f"Reusing independently trained {dataset} teacher, seed {seed}; original protocol verified")
        return model.eval(), {**checkpoint["info"], "reused_teacher": True}
    is_cifar = dataset == "cifar100"
    if is_cifar:
        optimizer = torch.optim.SGD(model.parameters(), lr=0.1, momentum=0.9, weight_decay=5e-4)
    else:
        optimizer = torch.optim.AdamW(model.parameters(), lr=0.002, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    amp = device.type == "cuda"
    try:
        scaler = torch.amp.GradScaler("cuda", enabled=amp)
    except (AttributeError, TypeError):
        scaler = torch.cuda.amp.GradScaler(enabled=amp)
    best_metric = (-float("inf"), -float("inf"))
    best_state = cpu_state(model)
    start = 0
    elapsed_before = 0.0
    if path.exists():
        checkpoint = local_torch_load(path, device)
        if checkpoint["fingerprint"] != cfg.fingerprint():
            raise ValueError("Teacher checkpoint/configuration mismatch")
        if checkpoint.get("complete", False):
            model.load_state_dict(checkpoint["best_state"])
            return model.eval(), checkpoint["info"]
        model.load_state_dict(checkpoint["model"])
        optimizer.load_state_dict(checkpoint["optimizer"])
        scheduler.load_state_dict(checkpoint["scheduler"])
        scaler.load_state_dict(checkpoint["scaler"])
        best_state, best_metric = checkpoint["best_state"], tuple(checkpoint["best_metric"])
        start, elapsed_before = checkpoint["epoch"], checkpoint["elapsed"]
        torch.set_rng_state(checkpoint["torch_rng"].cpu())
        if amp and checkpoint.get("cuda_rng") is not None:
            torch.cuda.set_rng_state_all(checkpoint["cuda_rng"])
        np.random.set_state(checkpoint["numpy_rng"])
        random.setstate(checkpoint["python_rng"])
        log(f"Resuming {dataset} teacher, seed {seed}, epoch {start}/{epochs}")
    val_loader = make_loader(data.val, cfg)
    started = time.monotonic()
    for epoch in range(start, epochs):
        model.train()
        loader = make_loader(data.train, cfg, shuffle=True, seed=seed * 1000 + epoch)
        training_loss = 0.0
        seen = 0
        for x, y in loader:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=amp):
                logits = model(x)
                loss = F.cross_entropy(logits, y)
            if not torch.isfinite(loss):
                raise FloatingPointError(f"Teacher training diverged: {dataset}, seed {seed}")
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 10)
            scaler.step(optimizer)
            scaler.update()
            training_loss += float(loss.detach()) * len(y)
            seen += len(y)
        scheduler.step()
        validation = evaluate(model, val_loader, device)
        metric = (validation["accuracy"], -validation["loss"])
        if metric > best_metric:
            best_metric, best_state = metric, cpu_state(model)
        elapsed = elapsed_before + time.monotonic() - started
        info = dict(epochs=epochs, best_validation_accuracy=best_metric[0], training_seconds=elapsed)
        checkpoint = dict(fingerprint=cfg.fingerprint(), epoch=epoch + 1, complete=epoch + 1 == epochs,
                          model=cpu_state(model), best_state=best_state, best_metric=best_metric,
                          optimizer=optimizer.state_dict(), scheduler=scheduler.state_dict(),
                          scaler=scaler.state_dict(), elapsed=elapsed, info=info,
                          torch_rng=torch.get_rng_state(),
                          cuda_rng=torch.cuda.get_rng_state_all() if amp else None,
                          numpy_rng=np.random.get_state(), python_rng=random.getstate())
        atomic_torch_save(checkpoint, path)
        log(f"{dataset}, seed {seed}, epoch {epoch + 1}/{epochs}: train CE={training_loss/seen:.4f}, "
            f"validation={validation['accuracy']:.2f}%, best={best_metric[0]:.2f}%")
    model.load_state_dict(best_state)
    return model.eval(), info


def dataset_tensors(dataset: Dataset, indices: np.ndarray) -> tuple[torch.Tensor, torch.Tensor]:
    examples = [dataset[int(i)] for i in indices]
    return torch.stack([e[0] for e in examples]), torch.tensor([e[1] for e in examples], dtype=torch.long)


def sensitivity_sketch(model: nn.Module, graph: WeightGraph, x: torch.Tensor,
                       cfg: Config, device: torch.device, seed: int,
                       return_vectors: bool = False) -> np.ndarray | tuple[np.ndarray,np.ndarray]:
    """Hutchinson sketch of centered-output Jacobian squared, in log-weight coordinates.

    Random independent output probes make batch-gradient squares an unbiased
    estimator before clipping/shrinkage. This is a Jacobian sketch, not an
    exact Fisher matrix. Probes are generated locally; input data are real.
    """
    model.eval()
    alpha = np.zeros(graph.n_weights, np.float64)
    vectors=[]
    generator = torch.Generator(device=device).manual_seed(seed + 80001)
    n = 0
    batch_size = min(64, cfg.batch_size)
    for batch_i, start in enumerate(range(0, len(x), batch_size)):
        if batch_i >= cfg.sketch_batches:
            break
        batch = x[start:start + batch_size].to(device)
        logits = model(batch)
        centered = logits - logits.mean(dim=1, keepdim=True)
        for probe_i in range(cfg.sketch_probes):
            vector=np.zeros(graph.n_weights,np.float32) if return_vectors else None
            model.zero_grad(set_to_none=True)
            probe = torch.randint(0, 2, logits.shape, device=device, generator=generator).float().mul_(2).sub_(1)
            probe /= math.sqrt(logits.shape[1])
            (centered * probe).sum().backward(retain_graph=probe_i + 1 < cfg.sketch_probes)
            for layer in graph.layers:
                m = module_at(model, layer.name)
                if m.weight.grad is None:
                    continue
                product = (m.weight.detach() * m.weight.grad).cpu().numpy().reshape(-1).astype(np.float64)
                alpha[layer.start:layer.stop] += product * product / cfg.sketch_probes
                if return_vectors:
                    vector[layer.start:layer.stop]=product
            if return_vectors:
                vectors.append(vector)
        n += len(batch)
    model.zero_grad(set_to_none=True)
    alpha *= math.log(2) ** 2 / max(n, 1)
    positive = alpha[alpha > 0]
    if len(positive) == 0:
        raise RuntimeError("All output-Jacobian sensitivities are zero")
    # A gauge-invariant scalar prior, rather than a gauge-dependent |w| prior.
    alpha += np.median(positive) * 1e-5
    if not np.isfinite(alpha).all():
        raise FloatingPointError("Invalid sensitivity sketch")
    if return_vectors:
        sketch=np.stack(vectors).astype(np.float32)*float(math.log(2)/math.sqrt(max(n,1)*cfg.sketch_probes))
        return alpha,sketch
    return alpha


def uniform_spec(weight: np.ndarray, bits: int, scale: np.ndarray | None = None) -> dict:
    shape = weight.shape
    flat = weight.reshape(shape[0], -1).astype(np.float64)
    qmax = 2 ** (bits - 1) - 1
    if scale is None:
        maximum = np.max(np.abs(flat), axis=1)
        best_error = np.full(shape[0], np.inf)
        best_scale = np.maximum(maximum / qmax, 1e-12)
        for fraction in (0.5, 0.65, 0.8, 0.9, 0.97, 1.0):
            candidate = np.maximum(maximum * fraction / qmax, 1e-12)
            code = np.clip(np.rint(flat / candidate[:, None]), -qmax, qmax)
            error = np.mean(np.square(code * candidate[:, None] - flat), axis=1)
            chosen = error < best_error
            best_error[chosen] = error[chosen]
            best_scale[chosen] = candidate[chosen]
        scale = best_scale.astype(np.float32)
    scale = np.asarray(scale, np.float32).reshape(-1)
    code = np.clip(np.rint(flat / scale[:, None]), -qmax, qmax).astype(np.int32)
    value = (code.astype(np.float32) * scale[:, None]).reshape(shape)
    return dict(kind="uniform", shape=list(shape), bits=bits, scale=scale,
                code=(code.reshape(-1) + qmax).astype(np.uint8), value=value)


def lsq_spec(weight: np.ndarray, bits: int, scale: np.ndarray | None = None) -> dict:
    """Weight-only LSQ quantizer (Esser et al., ICLR 2020), followed by common KD.

    This evaluates the published WEIGHT quantizer; it is not a reproduction
    of an activation-quantized LSQ network. One learnable step size per tensor.
    """
    qn,qp=-(2**(bits-1)),2**(bits-1)-1
    if scale is None:
        scale=np.asarray([max(2*float(np.mean(np.abs(weight)))/math.sqrt(qp),1e-12)],np.float32)
    scale=np.maximum(np.asarray(scale,np.float32).reshape(1),1e-12)
    code=np.clip(np.rint(weight.astype(np.float64)/float(scale[0])),qn,qp).astype(np.int32)
    return dict(kind="lsq",shape=list(weight.shape),bits=bits,scale=scale,zero_point=-qn,
                code=(code.reshape(-1)-qn).astype(np.uint8),value=(code.astype(np.float32)*scale[0]))


class WeightProjector(nn.Module):
    def __init__(self, spec: dict):
        super().__init__()
        self.kind = spec["kind"]
        if self.kind == "lsq":
            self.bits=spec["bits"]
            self.scale=nn.Parameter(torch.from_numpy(spec["scale"].copy()))
        elif self.kind == "uniform":
            self.bits = spec["bits"]
            self.register_buffer("scale", torch.from_numpy(spec["scale"].copy()))
        elif self.kind == "nested":
            self.exponent_bits,self.min_exp,self.block_size = spec["exponent_bits"],spec["min_exp"],spec["block_size"]
            depths=spec["group_depths"].copy()
            self.register_buffer("group_depths",torch.from_numpy(depths))
            expanded=np.repeat(depths,group_sizes(int(np.prod(spec["shape"])),self.block_size)).reshape(spec["shape"])
            self.register_buffer("weight_depths",torch.from_numpy(expanded))
            for depth in range(4):
                values,_,_=nested_codebook(self.min_exp,self.exponent_bits,depth)
                self.register_buffer(f"nested_values_{depth}",torch.from_numpy(values.astype(np.float32)))
                self.register_buffer(f"nested_midpoints_{depth}",torch.from_numpy(((values[:-1]+values[1:])/2).astype(np.float32)))
        else:
            self.exponent_bits, self.min_exp = spec["exponent_bits"], spec["min_exp"]
            self.register_buffer("mask", torch.from_numpy(spec["mask"].reshape(spec["shape"]).copy()))
            values, _, _ = dyadic_codebook(self.min_exp, self.exponent_bits)
            self.register_buffer("dyadic_values", torch.from_numpy(values.astype(np.float32)))
            self.register_buffer("dyadic_midpoints", torch.from_numpy(((values[:-1] + values[1:]) / 2).astype(np.float32)))

    def forward(self, weight: torch.Tensor) -> torch.Tensor:
        if self.kind == "lsq":
            qn,qp=-(2**(self.bits-1)),2**(self.bits-1)-1
            raw_scale=self.scale.clamp_min(1e-12)
            factor=1/math.sqrt(weight.numel()*qp)
            scale=(raw_scale-raw_scale*factor).detach()+raw_scale*factor
            normalized=(weight/scale).clamp(qn,qp)
            rounded=normalized+(normalized.round()-normalized).detach()
            return rounded*scale
        if self.kind == "uniform":
            qmax = 2 ** (self.bits - 1) - 1
            scale = self.scale.reshape((-1,) + (1,) * (weight.ndim - 1))
            quantized = torch.round(weight / scale).clamp(-qmax, qmax) * scale
        elif self.kind == "nested":
            mag=weight.abs().contiguous()
            q=torch.zeros_like(mag)
            for depth in range(4):
                values=getattr(self,f"nested_values_{depth}")
                midpoints=getattr(self,f"nested_midpoints_{depth}")
                ids=torch.bucketize(mag,midpoints,right=False)
                q=torch.where(self.weight_depths==depth,values[ids],q)
            quantized=weight.sign()*q
        else:
            magnitude = weight.abs()
            maximum = self.min_exp + 2 ** self.exponent_bits - 2
            low_exp = torch.floor(torch.log2(magnitude.clamp_min(1e-30))).clamp(self.min_exp, maximum)
            high_exp = (low_exp + 1).clamp(self.min_exp, maximum)
            low, high = torch.exp2(low_exp), torch.exp2(high_exp)
            base = torch.where((magnitude - low).abs() <= (magnitude - high).abs(), low, high)
            base = torch.where(magnitude < 0.5 * 2.0 ** self.min_exp, torch.zeros_like(base), base)
            ids = torch.bucketize(magnitude.contiguous(), self.dyadic_midpoints, right=False)
            value = torch.where(self.mask, self.dyadic_values[ids], base)
            quantized = weight.sign() * value
        return weight + (quantized - weight).detach()  # straight-through derivative

    def export(self, weight: torch.Tensor) -> dict:
        w = weight.detach().cpu().numpy()
        if self.kind == "lsq":
            return lsq_spec(w,self.bits,self.scale.detach().cpu().numpy())
        if self.kind == "uniform":
            return uniform_spec(w, self.bits, self.scale.detach().cpu().numpy())
        if self.kind == "nested":
            return nested_spec(w,np.ones(w.size),self.group_depths.detach().cpu().numpy(),
                               self.exponent_bits,self.block_size,self.min_exp)
        return pot_spec(w, np.ones(w.size), self.mask.detach().cpu().numpy().reshape(-1),
                        self.exponent_bits, self.min_exp)


class QuantConv(nn.Conv2d):
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self._conv_forward(x, self.projector(self.weight), self.bias)


class QuantLinear(nn.Linear):
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return F.linear(x, self.projector(self.weight), self.bias)


def install_quantizers(model: nn.Module, specs: dict) -> nn.Module:
    student = copy.deepcopy(model)
    for name, spec in specs.items():
        old = module_at(student, name)
        if isinstance(old, nn.Conv2d):
            new = QuantConv(old.in_channels, old.out_channels, old.kernel_size, old.stride,
                            old.padding, old.dilation, old.groups, old.bias is not None,
                            old.padding_mode, device=old.weight.device, dtype=old.weight.dtype)
        elif isinstance(old, nn.Linear):
            new = QuantLinear(old.in_features, old.out_features, old.bias is not None,
                              device=old.weight.device, dtype=old.weight.dtype)
        else:
            raise TypeError(name)
        with torch.no_grad():
            new.weight.copy_(old.weight)
            if new.bias is not None:
                new.bias.copy_(old.bias)
        new.projector = WeightProjector(spec).to(old.weight.device)
        replace_module(student, name, new)
    return student


def materialize_student(student: nn.Module, arch: dict, names: list[str]) -> tuple[nn.Module, dict]:
    # Reconstruct using the authoritative CPU codec, avoiding GPU/CPU rounding
    # differences at a threshold. The test set sees this actual packed model.
    architecture = {**arch, "folded": True}
    model = build_model(architecture).eval()
    specs = {}
    state = {}
    for name in names:
        m = module_at(student, name)
        spec = m.projector.export(m.weight)
        specs[name] = spec
        state[name + ".weight"] = torch.from_numpy(spec["value"].copy())
        if m.bias is not None:
            state[name + ".bias"] = m.bias.detach().cpu().clone()
    model.load_state_dict(state, strict=True)
    return model, specs


def pack_uint(values: np.ndarray, bits: int) -> bytes:
    v = np.asarray(values, np.uint64).reshape(-1)
    if not 1 <= bits <= 16 or (len(v) and int(v.max()) >= 2 ** bits):
        raise ValueError("Integer code does not fit its stored bit width")
    if len(v) == 0:
        return b""
    b = ((v[:, None] >> np.arange(bits, dtype=np.uint64)) & 1).astype(np.uint8).reshape(-1)
    return np.packbits(b, bitorder="little").tobytes()


def unpack_uint(data: bytes, count: int, bits: int) -> np.ndarray:
    if len(data) != (count * bits + 7) // 8:
        raise ValueError("Packed integer payload length mismatch")
    if count == 0:
        return np.empty(0, np.uint64)
    b = np.unpackbits(np.frombuffer(data, np.uint8), bitorder="little")[:count * bits].reshape(count, bits)
    return (b.astype(np.uint64) @ (np.uint64(1) << np.arange(bits, dtype=np.uint64)))


def depth_payload(depths: np.ndarray) -> tuple[str,bytes,int]:
    depths=np.asarray(depths,np.uint8)
    if len(depths) and np.all(depths==depths[0]):
        return "constant",b"",int(depths[0])
    raw=pack_uint(depths,2)
    compressed=zlib.compress(raw,level=9)
    return ("zlib",compressed,0) if len(compressed)<len(raw) else ("packed",raw,0)


def encode_model(model: nn.Module, arch: dict, specs: dict | None = None,
                 size_only: bool = False) -> bytes | int:
    parts = []
    offset = 0
    records = []
    def add(data: bytes = b"", length: int | None = None) -> dict:
        nonlocal offset
        length=len(data) if length is None else length
        description = dict(offset=offset, length=length)
        if not size_only:
            if length != len(data):
                raise AssertionError("A real payload must match its byte accounting")
            parts.append(data)
        offset += length
        return description
    for key, tensor in model.state_dict().items():
        name = key.removesuffix(".weight")
        spec = (specs or {}).get(name) if key.endswith(".weight") else None
        record = dict(key=key, shape=list(tensor.shape))
        if spec is None:
            if tensor.dtype != torch.float32:
                raise TypeError(f"Unexpected unencoded state dtype: {key}, {tensor.dtype}")
            data=b"" if size_only else tensor.detach().cpu().numpy().astype("<f4").tobytes()
            record.update(kind="float32", data=add(data,tensor.numel()*4))
        elif spec["kind"] in ("uniform","lsq"):
            data=b"" if size_only else pack_uint(spec["code"],spec["bits"])
            scales=b"" if size_only else spec["scale"].astype("<f4").tobytes()
            record.update(kind="uniform", bits=spec["bits"],
                          data=add(data,(tensor.numel()*spec["bits"]+7)//8),
                          scales=add(scales,len(spec["scale"])*4))
            if spec["kind"]=="lsq":
                record["zero_point"]=spec["zero_point"]
        elif spec["kind"] == "nested":
            bits=spec["exponent_bits"]
            depths=spec["group_depths"]
            sizes=group_sizes(tensor.numel(),spec["block_size"])
            mode,ddata,constant=depth_payload(depths)
            integer=None if size_only else (spec["base_code"].astype(np.uint16)<<1)|spec["sign"].astype(np.uint16)
            data=b"" if size_only else pack_uint(integer,bits+1)
            record.update(kind="nested",exponent_bits=bits,min_exp=spec["min_exp"],block_size=spec["block_size"],
                          data=add(data,(tensor.numel()*(bits+1)+7)//8),depth_mode=mode,
                          constant_depth=constant,depths=add(ddata),planes=[])
            wdepth=None if size_only else np.repeat(depths,sizes)
            for plane in range(1,4):
                count=int(sizes[depths>=plane].sum())
                plane_data=b""
                if not size_only:
                    ids=wdepth>=plane
                    values=(spec["mantissa"][ids]>>(wdepth[ids]-plane))&1
                    plane_data=pack_uint(values,1)
                record["planes"].append(add(plane_data,(count+7)//8))
        else:
            bits = spec["exponent_bits"]
            integer = None if size_only else (spec["base_code"].astype(np.uint16) << 1) | spec["sign"].astype(np.uint16)
            mask = spec["mask"]
            raw_mask = np.packbits(mask, bitorder="little").tobytes()
            compressed_mask = zlib.compress(raw_mask, level=9)
            if not mask.any():
                mask_mode, stored_mask = "none", b""
            elif len(compressed_mask) < len(raw_mask):
                mask_mode, stored_mask = "zlib", compressed_mask
            else:
                mask_mode, stored_mask = "bitmap", raw_mask
            record.update(kind="pot", exponent_bits=bits, min_exp=spec["min_exp"],
                          data=add(b"" if size_only else pack_uint(integer,bits+1),(tensor.numel()*(bits+1)+7)//8), mask_mode=mask_mode,
                          mask=add(stored_mask), count_terms=int(mask.sum()),
                          tags=add(b"" if size_only else pack_uint(spec["tag"][mask],TERM_BITS),(int(mask.sum())*TERM_BITS+7)//8))
        records.append(record)
    header = dict(version=VERSION, architecture={**arch,"folded":True},records=records)
    header_bytes = json.dumps(header, separators=(",", ":"), sort_keys=True).encode()
    header_bytes=zlib.compress(header_bytes,level=9)
    # The checksum is stored OUTSIDE the compressed header, so changing weight
    # codes cannot change the file length during STE refinement.
    if size_only:
        return 12+len(header_bytes)+offset+32
    payload=b"".join(parts)
    return MAGIC2+struct.pack("<I",len(header_bytes))+header_bytes+payload+hashlib.sha256(payload).digest()


def decode_model(blob: bytes) -> tuple[nn.Module, dict]:
    if blob[:8] not in (MAGIC,MAGIC2):
        raise ValueError("Invalid CAPQ model signature")
    n_header = struct.unpack("<I", blob[8:12])[0]
    if blob[:8] == MAGIC2:
        header=json.loads(zlib.decompress(blob[12:12+n_header]))
        payload=blob[12+n_header:-32]
        checksum_ok=hashlib.sha256(payload).digest()==blob[-32:]
    else:
        header=json.loads(blob[12:12+n_header])
        payload=blob[12+n_header:]
        checksum_ok=hashlib.sha256(payload).hexdigest()==header["payload_sha256"]
    if not checksum_ok:
        raise ValueError("Compressed model checksum mismatch")
    def get(part: dict) -> bytes:
        a, n = part["offset"], part["length"]
        if a < 0 or n < 0 or a + n > len(payload):
            raise ValueError("Invalid payload boundaries")
        return payload[a:a + n]
    state = {}
    for record in header["records"]:
        shape, kind = tuple(record["shape"]), record["kind"]
        count = int(np.prod(shape))
        if kind == "float32":
            value = np.frombuffer(get(record["data"]), dtype="<f4").copy().reshape(shape)
        elif kind == "uniform":
            code = unpack_uint(get(record["data"]), count, record["bits"]).astype(np.int32)
            qmax = 2 ** (record["bits"] - 1) - 1
            scale = np.frombuffer(get(record["scales"]), dtype="<f4").copy()
            value = ((code-record.get("zero_point",qmax)).reshape(shape[0], -1).astype(np.float32) * scale[:, None]).reshape(shape)
        elif kind == "nested":
            raw=unpack_uint(get(record["data"]),count,record["exponent_bits"]+1).astype(np.int32)
            code,sign=raw>>1,raw&1
            sizes=group_sizes(count,record["block_size"])
            if record["depth_mode"] == "constant":
                depths=np.full(len(sizes),record["constant_depth"],np.uint8)
            else:
                data=get(record["depths"])
                if record["depth_mode"] == "zlib":
                    data=zlib.decompress(data)
                elif record["depth_mode"] != "packed":
                    raise ValueError("Invalid group-depth codec")
                depths=unpack_uint(data,len(sizes),2).astype(np.uint8)
            if np.any(depths>3):
                raise ValueError("Invalid group mantissa depth")
            wdepth=np.repeat(depths,sizes)
            mantissa=np.zeros(count,np.uint8)
            for plane in range(1,4):
                ids=wdepth>=plane
                values=unpack_uint(get(record["planes"][plane-1]),int(ids.sum()),1).astype(np.uint8)
                mantissa[ids]=(mantissa[ids]<<1)|values
            base=np.exp2(code+record["min_exp"]-1).astype(np.float32)
            base[code==0]=0
            coefficient=1+mantissa.astype(np.float32)/np.exp2(wdepth.astype(np.float32))
            value=(np.where(sign!=0,np.float32(-1),np.float32(1))*base*coefficient).reshape(shape)
        elif kind == "pot":
            raw = unpack_uint(get(record["data"]), count, record["exponent_bits"] + 1).astype(np.int32)
            code, sign = raw >> 1, raw & 1
            base = np.exp2(code + record["min_exp"] - 1).astype(np.float32)
            base[code == 0] = 0
            if record["mask_mode"] == "none":
                mask = np.zeros(count, bool)
            else:
                data = get(record["mask"])
                if record["mask_mode"] == "zlib":
                    data = zlib.decompress(data)
                mask = np.unpackbits(np.frombuffer(data, np.uint8), bitorder="little")[:count].astype(bool)
            if int(mask.sum()) != record["count_terms"]:
                raise ValueError("Exception mask population mismatch")
            tags = unpack_uint(get(record["tags"]), int(mask.sum()), TERM_BITS).astype(np.int32)
            if len(tags) and int(tags.max()) >= len(COEFF):
                raise ValueError("Invalid mantissa tag")
            coefficient = np.ones(count, np.float32)
            coefficient[mask] = COEFF[tags]
            value = (np.where(sign != 0, np.float32(-1), np.float32(1)) * base * coefficient).reshape(shape)
        else:
            raise ValueError(kind)
        state[record["key"]] = torch.from_numpy(value.copy())
    model = build_model(header["architecture"]).eval()
    model.load_state_dict(state, strict=True)
    return model, header


def assert_roundtrip(model: nn.Module, blob: bytes) -> nn.Module:
    restored, _ = decode_model(blob)
    for key, value in model.state_dict().items():
        if not torch.equal(value.detach().cpu().contiguous().view(torch.int32),
                           restored.state_dict()[key].contiguous().view(torch.int32)):
            raise AssertionError(f"Codec is not lossless with respect to quantized tensor {key}")
    return restored


def refine_student(student: nn.Module, teacher: nn.Module, x: torch.Tensor, y: torch.Tensor,
                   selection_loader: DataLoader, seed: int, cfg: Config, device: torch.device,
                   checkpoint_path: Path, arch: dict, names: list[str]) -> tuple[nn.Module, dict, dict]:
    teacher.eval()
    student.to(device)
    with torch.no_grad():
        targets = torch.cat([teacher(a.to(device)).cpu() for a in x.split(cfg.batch_size)])
    optimizer = torch.optim.AdamW(student.parameters(), lr=cfg.refinement_lr, weight_decay=0)
    initial = evaluate(student, selection_loader, device)
    best_metric = (initial["accuracy"], -initial["loss"])
    best_state = cpu_state(student)
    start = 0
    before = 0.0
    best_step = 0
    if checkpoint_path.exists():
        checkpoint = local_torch_load(checkpoint_path, device)
        if checkpoint["fingerprint"] != cfg.fingerprint():
            raise ValueError("Refinement checkpoint/configuration mismatch")
        if checkpoint.get("complete", False):
            student.load_state_dict(checkpoint["best_state"])
            materialized, specs = materialize_student(student, arch, names)
            return materialized, specs, checkpoint["info"]
        student.load_state_dict(checkpoint["model"])
        optimizer.load_state_dict(checkpoint["optimizer"])
        initial = checkpoint["initial"]
        best_state, best_metric = checkpoint["best_state"], tuple(checkpoint["best_metric"])
        start, before = checkpoint["step"], checkpoint["elapsed"]
        best_step = checkpoint["info"]["selected_step"]
    started = time.monotonic()
    for step in range(start, cfg.refinement_steps):
        student.train()
        # Stateless batch sampling makes an interrupted run resume exactly.
        rng = np.random.default_rng(seed * 1000003 + step)
        ids = rng.choice(len(x), size=min(cfg.batch_size, len(x)), replace=False)
        batch = x[ids].to(device)
        label = y[ids].to(device)
        target = targets[ids].to(device)
        optimizer.zero_grad(set_to_none=True)
        logits = student(batch)
        temperature = cfg.kd_temperature
        kl = F.kl_div(F.log_softmax(logits / temperature, dim=1),
                      F.softmax(target / temperature, dim=1), reduction="batchmean") * temperature**2
        ce = F.cross_entropy(logits, label)
        loss = cfg.kd_weight * kl + (1 - cfg.kd_weight) * ce
        if not torch.isfinite(loss):
            raise FloatingPointError("Compressed-model refinement produced non-finite loss")
        loss.backward()
        torch.nn.utils.clip_grad_norm_(student.parameters(), 5)
        optimizer.step()
        if (step + 1) % cfg.eval_interval == 0 or step + 1 == cfg.refinement_steps:
            validation = evaluate(student, selection_loader, device)
            metric = (validation["accuracy"], -validation["loss"])
            if metric > best_metric:
                best_metric, best_state, best_step = metric, cpu_state(student), step + 1
            elapsed = before + time.monotonic() - started
            info = dict(initial_validation_accuracy=initial["accuracy"],
                        selected_validation_accuracy=best_metric[0], selected_step=best_step,
                        refinement_steps=cfg.refinement_steps, refinement_seconds=elapsed)
            atomic_torch_save(dict(fingerprint=cfg.fingerprint(), complete=step + 1 == cfg.refinement_steps,
                                  step=step + 1, model=cpu_state(student), optimizer=optimizer.state_dict(),
                                  best_state=best_state, best_metric=best_metric, initial=initial,
                                  elapsed=elapsed, info=info), checkpoint_path)
            log(f"Refinement {step + 1}/{cfg.refinement_steps}: validation={validation['accuracy']:.2f}%, "
                f"selected={best_metric[0]:.2f}% (initial {initial['accuracy']:.2f}%)")
    student.load_state_dict(best_state)
    model, specs = materialize_student(student, arch, names)
    info = dict(initial_validation_accuracy=initial["accuracy"], selected_validation_accuracy=best_metric[0],
                selected_step=best_step, refinement_steps=cfg.refinement_steps,
                refinement_seconds=before + time.monotonic() - started)
    return model, specs, info


@torch.no_grad()
def verify_teacher_and_graph(original: nn.Module, folded: nn.Module, graph: WeightGraph,
                             real_x: torch.Tensor, cfg: Config, seed: int,
                             device: torch.device) -> dict:
    x = real_x[:min(12, len(real_x))].to(device)
    original.eval(); folded.eval()
    y_original, y_folded = original(x), folded(x)
    torch.testing.assert_close(y_original, y_folded, rtol=2e-4, atol=2e-4)
    s = np.random.default_rng(seed + 177).uniform(-1, 1, graph.n_nodes)
    reparameterized = graph.apply(s).to(device).eval()
    y_scaled = reparameterized(x)
    torch.testing.assert_close(y_folded, y_scaled, rtol=2e-4, atol=2e-4)
    v = graph.x - np.rint(graph.x)
    projected = graph.project_residual(v)
    orthogonality = np.linalg.norm(graph.rhs(projected)) / max(np.linalg.norm(graph.rhs(v)), 1e-12)
    assert orthogonality < 2e-6, f"Weighted cycle projection residual {orthogonality}"
    e = int(np.argmax(graph.alpha))
    mu = math.log2(1.125)
    changed = v.copy(); changed[e] -= mu
    before = float(np.dot(graph.alpha, projected**2))
    after_r = graph.project_residual(changed)
    after = float(np.dot(graph.alpha, after_r**2))
    predicted = before - 2 * mu * graph.alpha[e] * projected[e] + mu**2 * graph.cycle_curvature[e]
    assert abs(after - predicted) <= 1e-7 * max(before, 1), "Incorrect graph marginal-gain formula"
    delta = np.zeros(graph.n_weights)
    start = (e//cfg.compact_block_size)*cfg.compact_block_size
    delta[start:min(start+cfg.compact_block_size, graph.n_weights)] = mu
    block_rhs = graph.rhs(delta)
    block_gain = (2*np.dot(graph.alpha*projected, delta)-np.dot(graph.alpha, delta**2)
                  + block_rhs@sla.cho_solve(graph.factor, block_rhs, check_finite=False))
    actual_block_after = graph.project_residual(v-delta)
    block_after = float(np.dot(graph.alpha, actual_block_after**2))
    assert abs(block_after-(before-block_gain)) <= 1e-7*max(before, 1), "Incorrect full block cycle marginal"
    diamonds = 0
    for layer in graph.layers:
        m = module_at(folded, layer.name)
        if m.weight.shape[0] < 2 or m.weight.shape[1] < 2:
            continue
        w = m.weight.detach().cpu().numpy().reshape(m.weight.shape[0], m.weight.shape[1], -1)
        a = np.abs(w[:2, :2, 0].reshape(-1)).astype(np.float64)
        if (a <= 1e-12).any():
            continue
        cx = np.log2(a)
        c = np.array([1, -1, -1, 1])
        extended = np.concatenate([s, [0]])
        shifts = extended[layer.heads[:2], None] - extended[layer.tails[None, :2]]
        rho = float(c @ cx)
        rho_after = float(c @ (cx + shifts.reshape(-1)))
        assert abs(rho - rho_after) < 1e-8, "Cycle invariant changed under admissible rescaling"
        r = cx + shifts.reshape(-1) - np.rint(cx + shifts.reshape(-1))
        bound = (rho - round(rho))**2 / 4
        assert float(r @ r) + 1e-10 >= bound, "Cycle distortion lower bound violated"
        diamonds += 1
    assert diamonds > 0, "No valid real-weight diamond checked"
    fp_blob = encode_model(folded.cpu(), graph.arch)
    decoded = assert_roundtrip(folded.cpu(), fp_blob).to(device)
    torch.testing.assert_close(decoded(x), y_folded, rtol=2e-4, atol=2e-4)
    folded.to(device)
    return dict(folding_max_error=float((y_original - y_folded).abs().max()),
                gauge_max_error=float((y_folded - y_scaled).abs().max()),
                projection_relative_residual=float(orthogonality), diamonds_checked=diamonds,
                marginal_gain_verified=True, full_block_marginal_verified=True, float32_codec_verified=True)


def save_rows(rows: list[dict], path: Path) -> None:
    temporary = path.with_suffix(".tmp.csv")
    pd.DataFrame(rows).to_csv(temporary, index=False)
    os.replace(temporary, path)


def condition_key(dataset: str, seed: int, method: str, budget: str) -> str:
    return f"{dataset}|{seed}|{method}|{budget}"


def condition_list(cfg: Config) -> list[tuple[str, str, float | int]]:
    conditions: list[tuple[str, str, float | int]] = [("PoT", "p=0", 0.0)]
    conditions += [("Uniform+KD", f"{b}-bit", b) for b in cfg.uniform_bits]
    if cfg.protocol == "rd":
        conditions += [("LSQ-W+KD",f"{b}-bit",b) for b in cfg.uniform_bits]
        for bits in cfg.uniform_bits:
            conditions += [(method,f"bytes={bits}-bit",bits) for method in RD_MAIN_METHODS]
        middle=cfg.uniform_bits[len(cfg.uniform_bits)//2]
        conditions += [(method,f"bytes={middle}-bit",middle) for method in RD_ABLATIONS]
    else:
        for p in cfg.fractions:
            conditions += [(method, f"p={p:g}", p) for method in MAIN_METHODS]
        middle = cfg.fractions[len(cfg.fractions) // 2]
        conditions += [(method, f"p={middle:g}", middle) for method in ABLATIONS]
    if cfg.methods:
        conditions = [condition for condition in conditions if condition[0] in cfg.methods]
    return conditions


def report_results(rows: list[dict], cfg: Config, root: Path, final: bool = True) -> pd.DataFrame:
    frame = pd.DataFrame(rows)
    if frame.empty:
        return frame
    group_keys = ["dataset", "method", "budget"]
    complete_keys = []
    for key, group in frame.groupby(group_keys, sort=False):
        seeds = group.seed.astype(int).tolist()
        if len(seeds) == 5 and set(seeds) == set(SEEDS):
            complete_keys.append(key)
        elif final:
            raise AssertionError(f"Incomplete/duplicated five-seed group: {key}, seeds={seeds}")
    if not complete_keys:
        return pd.DataFrame()
    keep = [tuple(a) in complete_keys for a in frame[group_keys].itertuples(index=False, name=None)]
    frame = frame.loc[keep].copy()
    metrics = ["accuracy", "initial_accuracy", "bytes", "bits_per_parameter", "compression_ratio",
               "compression_seconds", "used_nonpot_fraction","teacher_kl","logit_nmse","initial_teacher_kl","initial_logit_nmse"]
    metrics=[metric for metric in metrics if metric in frame]
    agg = frame.groupby(group_keys, sort=False)[metrics].agg(["mean", "std"])
    agg.columns = [f"{a}_{b}" for a, b in agg.columns]
    summary = agg.reset_index()
    summary["n_seeds"] = 5
    summary.to_csv(root / "five_seed_summary.csv", index=False)
    fmt = lambda row, metric, digits=2: f"{row[metric + '_mean']:.{digits}f} ± {row[metric + '_std']:.{digits}f}"
    visible = pd.DataFrame([dict(Dataset=r["dataset"], Method=r["method"], Budget=r["budget"],
                                **{"Accuracy (%)": fmt(r, "accuracy"),
                                   "Initial accuracy (%)": fmt(r, "initial_accuracy"),
                                   "Stored bits/parameter": fmt(r, "bits_per_parameter", 3),
                                   "Compression (×)": fmt(r, "compression_ratio", 3)})
                            for _, r in summary.iterrows()])
    if "teacher_kl_mean" in summary:
        visible["Teacher KL ×1000"]=[f"{1000*r['teacher_kl_mean']:.3f} ± {1000*r['teacher_kl_std']:.3f}" for _,r in summary.iterrows()]
        visible["Logit NMSE"]=[f"{r['logit_nmse_mean']:.4g} ± {r['logit_nmse_std']:.4g}" for _,r in summary.iterrows()]
    print("\nFIVE-SEED RESULTS: mean ± sample SD (ddof=1); no incomplete groups\n")
    with pd.option_context("display.max_rows", 100, "display.max_columns", 10, "display.width", 180):
        print(visible.to_string(index=False))
    try:
        from IPython import get_ipython
        from IPython.display import display
        if get_ipython() is not None:
            display(visible)
    except ImportError:
        pass
    selected_ablations=RD_ABLATIONS if cfg.protocol=="rd" else ABLATIONS
    main_summary = summary[~summary.method.isin(selected_ablations)]
    datasets = list(dict.fromkeys(main_summary.dataset))
    middle = (f"bytes={cfg.uniform_bits[len(cfg.uniform_bits)//2]}-bit" if cfg.protocol=="rd" else
              f"p={cfg.fractions[len(cfg.fractions)//2]:g}")
    table_summary = main_summary[(main_summary.budget == middle) |
                                 main_summary.method.isin(("FP32", "PoT", "Uniform+KD","LSQ-W+KD"))]
    pairs = list(dict.fromkeys(zip(table_summary.method, table_summary.budget)))
    dataset_labels = {"cifar100": "CIFAR-100", "speechcommands": "Speech-10",
                      "fashionmnist": "Fashion", "digits": "Digits"}
    table_rows = []
    for method, budget in pairs:
        row = {"Method": method, "Budget": budget}
        for dataset in datasets:
            sub = main_summary[(main_summary.dataset == dataset) & (main_summary.method == method) &
                               (main_summary.budget == budget)]
            if len(sub) == 1:
                r = sub.iloc[0]
                label = dataset_labels[dataset]
                row[f"{label} acc. (%)"] = fmt(r, "accuracy")
                row[f"{label} bits/param"] = fmt(r, "bits_per_parameter", 3)
                if "teacher_kl_mean" in r:
                    row[f"{label} teacher KL ×1000"]=f"{1000*r['teacher_kl_mean']:.3f} ± {1000*r['teacher_kl_std']:.3f}"
        table_rows.append(row)
    main_table = pd.DataFrame(table_rows)
    main_table.to_csv(root / "spl_table_1.csv", index=False)
    cell_latex = lambda x: str(x).replace("±", r"$\pm$").replace("_", r"\_")
    clean_table = main_table.fillna("—")
    mapper = clean_table.map if hasattr(clean_table, "map") else clean_table.applymap
    latex_table = mapper(cell_latex)
    (root / "spl_table_1.tex").write_text(latex_table.to_latex(index=False, escape=False), encoding="utf-8")
    ablation_methods = (("Local-RD","Static-RD","Diagonal-RD","RD-no-gauge","CAPQ-RD") if cfg.protocol=="rd" else
                       ("Sensitivity", "CAPQ-no-gauge", "CAPQ-no-sensitivity", "CAPQ", "CAPQ-compact"))
    reference_method="Local-RD" if cfg.protocol=="rd" else "Sensitivity"
    ablation_rows = []
    for dataset in datasets:
        subset = frame[(frame.dataset == dataset) & (frame.budget == middle)]
        reference = subset[subset.method == reference_method].set_index("seed").accuracy
        for method in ablation_methods:
            a = subset[subset.method == method].set_index("seed").sort_index()
            if len(a) != 5 or len(reference) != 5:
                continue
            delta = a.accuracy - reference.reindex(a.index)
            ablation_rows.append(dict(Dataset=dataset, Method=method,
                                      **{"Accuracy (%)": f"{a.accuracy.mean():.2f} ± {a.accuracy.std(ddof=1):.2f}",
                                         f"Paired gain over {reference_method} (pp)": f"{delta.mean():+.2f} ± {delta.std(ddof=1):.2f}",
                                         "Stored bits/parameter": f"{a.bits_per_parameter.mean():.3f} ± {a.bits_per_parameter.std(ddof=1):.3f}"}))
    ablation_table = pd.DataFrame(ablation_rows)
    ablation_table.to_csv(root / "spl_table_2.csv", index=False)
    if len(ablation_table):
        mapper = ablation_table.map if hasattr(ablation_table, "map") else ablation_table.applymap
        tex = mapper(cell_latex)
        (root / "spl_table_2.tex").write_text(tex.to_latex(index=False, escape=False), encoding="utf-8")
        print("\nMATCHED-CODEC ABLATION: paired gains use the SAME five training seeds\n")
        print(ablation_table.to_string(index=False))
    plt.rcParams.update({"font.size": 9, "axes.labelsize": 9, "legend.fontsize": 7,
                         "figure.dpi": 140, "savefig.dpi": 300, "pdf.fonttype": 42,
                         "ps.fonttype": 42, "axes.spines.top": False, "axes.spines.right": False})
    fig, axes = plt.subplots(1, len(datasets), figsize=(3.5 * len(datasets), 3.1), squeeze=False)
    colors = {"PoT": "#777777", "Uniform+KD": "#1261A0", "CLE+local": "#7D3C98",
              "Sensitivity": "#E67E22", "CAPQ": "#138D75", "CAPQ-compact":"#C0392B",
              "Local-RD":"#E67E22","CAPQ-RD":"#138D75","LSQ-W+KD":"#7D3C98"}
    for ax, dataset in zip(axes[0], datasets):
        subset = main_summary[main_summary.dataset == dataset]
        figure_methods=(("PoT","Uniform+KD","LSQ-W+KD","Local-RD","CAPQ-RD") if cfg.protocol=="rd" else
                        ("PoT", "Uniform+KD", "CLE+local", "Sensitivity", "CAPQ", "CAPQ-compact"))
        for method in figure_methods:
            a = subset[subset.method == method].sort_values("bits_per_parameter_mean")
            if len(a):
                ax.errorbar(a.bits_per_parameter_mean, a.accuracy_mean,
                            xerr=a.bits_per_parameter_std, yerr=a.accuracy_std,
                            marker="o", markersize=3.5, capsize=2, linewidth=1.2,
                            label=method, color=colors[method])
        reference = subset[subset.method == "FP32"]
        if len(reference):
            mean, std = float(reference.accuracy_mean.iloc[0]), float(reference.accuracy_std.iloc[0])
            ax.axhline(mean, ls="--", color="black", lw=0.8, label="FP32")
            ax.axhspan(mean - std, mean + std, color="black", alpha=0.06)
        ax.set_title(dataset + (" — verification" if dataset == "digits" or cfg.smoke else ""))
        ax.set_xlabel("Stored bits / parameter (all bytes)")
        ax.set_ylabel("Test accuracy (%)")
        ax.grid(alpha=0.2)
    fig.suptitle("Five seeds: mean ± sample SD", fontsize=10)
    handles, labels = axes[0, 0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="lower center", bbox_to_anchor=(0.5, 0.0),
               ncol=4, frameon=False, fontsize=7)
    fig.tight_layout(rect=(0, 0.17, 1, 1))
    fig.savefig(root / "spl_figure_1.png", bbox_inches="tight")
    fig.savefig(root / "spl_figure_1.pdf", bbox_inches="tight")
    plt.show()
    plt.close(fig)
    fig, axes = plt.subplots(1, len(datasets), figsize=(3.5 * len(datasets), 2.8), squeeze=False)
    for ax, dataset in zip(axes[0], datasets):
        subset = frame[(frame.dataset == dataset) & (frame.budget == middle)]
        labels, means, deviations = [], [], []
        for method in ablation_methods:
            a = subset[subset.method == method]
            if len(a) == 5:
                labels.append(method.replace("CAPQ-no-", "No "))
                means.append(float(a.accuracy.mean()))
                deviations.append(float(a.accuracy.std(ddof=1)))
        if means:
            positions = np.arange(len(means))
            ax.errorbar(positions, means, yerr=deviations, fmt="o", capsize=3,
                        color="#138D75", markersize=5)
            ax.set_xticks(positions, labels, rotation=25, ha="right", fontsize=7)
        ax.set_title(f"{dataset}, {middle}" + (" — verification" if dataset == "digits" or cfg.smoke else ""))
        ax.set_ylabel("Test accuracy (%)")
        ax.grid(axis="y", alpha=0.2)
    fig.suptitle("Matched codec: five-seed mean ± sample SD", fontsize=10)
    fig.tight_layout()
    fig.savefig(root / "spl_figure_2.png", bbox_inches="tight")
    fig.savefig(root / "spl_figure_2.pdf", bbox_inches="tight")
    plt.show()
    plt.close(fig)
    for dataset in datasets:
        budgets=([f"bytes={b}-bit" for b in cfg.uniform_bits] if cfg.protocol=="rd" else
                 [f"p={p:g}" for p in cfg.fractions])
        for budget in budgets:
            a = frame[(frame.dataset == dataset) & (frame.budget == budget)]
            full_method="CAPQ-RD" if cfg.protocol=="rd" else "CAPQ"
            full = a[a.method == full_method].set_index("seed").accuracy
            control = a[a.method == reference_method].set_index("seed").accuracy
            if len(full) == 5 and len(control) == 5:
                difference = full - control.reindex(full.index)
                log(f"{dataset}, {budget}: paired {full_method} − {reference_method} = "
                    f"{difference.mean():+.3f} ± {difference.std(ddof=1):.3f} percentage points")
    print("\nSPL core results: Table 1 (fixed middle budget), Figure 1 (budget sweep), "
          "Table 2 (ablation). Figure 2 is an optional view of Table 2, not an extra required experiment.\n"
          "Layout: page 1 problem + invariant; page 2 proposition + proof; "
          "page 3 algorithm + codec + setup; page 4 core results + conclusion; page 5 references.")
    log(f"Direct outputs saved in {root.resolve()} — no result archive created")
    return summary


def run_experiments(cfg: Config) -> None:
    if len(SEEDS) != 5 or len(set(SEEDS)) != 5:
        raise AssertionError("This runner requires exactly five distinct seeds")
    if not cfg.fractions or any(not 0 <= p <= 1 for p in cfg.fractions):
        raise ValueError("Invalid extra-term budgets")
    if not 2 <= cfg.exponent_bits <= 6:
        raise ValueError("Exponent bits must be between two and six")
    torch.set_num_threads(cfg.cpu_threads)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu") if cfg.device == "auto" else torch.device(cfg.device)
    if device.type == "cuda" and not torch.cuda.is_available():
        raise RuntimeError("CUDA was requested but is not available")
    root = Path(cfg.out) / cfg.fingerprint()
    root.mkdir(parents=True, exist_ok=True)
    protocol = dict(version=VERSION, teacher_revision=1, source_sha256=SOURCE_SHA256,
                    configuration=dataclasses.asdict(cfg), seeds=SEEDS,
                    seed_meaning="Five independent teacher initializations/training runs; paired compression conditions",
                    sample_std_ddof=1, split_seed=SPLIT_SEED,
                    checkpoint_selection="Validation accuracy, then validation cross-entropy; never the test set",
                    codec="Actual packed file bytes; all masks, biases, scales, metadata included",
                    portable_evaluation="Packed weights decoded to FP32; no measured hardware speedup claimed",
                    dataset_scope="Speech Commands v2 ten target words, excludes silence and unknown",
                    device=str(device), torch=torch.__version__, numpy=np.__version__,
                    cpu_backend="MKLDNN" if torch.backends.mkldnn.enabled else "portable torch convolution",
                    smoke=cfg.smoke, fingerprint=cfg.fingerprint())
    atomic_json(root / "protocol.json", protocol)
    result_path = root / "results.csv"
    rows = pd.read_csv(result_path).to_dict("records") if result_path.exists() else []
    completed = {condition_key(r["dataset"], int(r["seed"]), r["method"], r["budget"]) for r in rows}
    if len(completed) != len(rows):
        raise AssertionError("Duplicate result rows detected")
    log(f"{VERSION}; device={device}; seeds={SEEDS}; output={root.resolve()}")
    if not cfg.smoke and set(cfg.datasets)<=set(("cifar100","speechcommands")) and not cfg.methods:
        log("FULL CNN PAPER PROTOCOL")
    elif not cfg.smoke and set(cfg.datasets)=={"fashionmnist"}:
        log("FULL FASHION-MNIST EVALUATION: supplementary real-data evidence; full CNN benchmarks are separate")
    else:
        log("IMPLEMENTATION VERIFICATION: do not report this reduced protocol as the full SPL benchmark")
    loaders = {"cifar100": load_cifar100, "speechcommands": load_speech,
               "fashionmnist": load_fashion, "digits": load_real_digits}
    conditions = condition_list(cfg)
    diagnostics_path = root / "verification.json"
    diagnostics = json.loads(diagnostics_path.read_text()) if diagnostics_path.exists() else {}
    for dataset in cfg.datasets:
        expected = {condition_key(dataset, seed, method, budget) for seed in SEEDS
                    for method, budget in [("FP32", "FP32"), *[(m, b) for m, b, _ in conditions]]}
        if expected <= completed:
            log(f"All five seeds for {dataset} already complete; using measured saved rows")
            continue
        log(f"Preparing real dataset: {dataset}")
        data = loaders[dataset](cfg)
        atomic_json(root / f"dataset_{dataset}.json", data.dataset_info)
        log(json.dumps(data.dataset_info, ensure_ascii=False))
        select_ids = balanced_indices(data.val_labels, cfg.selection_size, SPLIT_SEED)
        selection_dataset = torch.utils.data.Subset(data.val, select_ids.tolist())
        selection_loader = make_loader(selection_dataset, cfg)
        test_loader = make_loader(data.test, cfg)
        for seed in SEEDS:
            seed_expected = {condition_key(dataset, seed, method, budget) for method, budget in
                             [("FP32", "FP32"), *[(m, b) for m, b, _ in conditions]]}
            if seed_expected <= completed:
                continue
            original, teacher_info = train_teacher(data, dataset, seed, cfg, root, device)
            teacher = fold_batchnorm(original).to(device).eval()
            calibration_ids = balanced_indices(data.labels, cfg.calibration_size, seed + 919)
            x_cal, y_cal = dataset_tensors(data.clean_train, calibration_ids)
            simple_graph = WeightGraph(teacher, data.architecture)
            if not cfg.skip_self_tests:
                diagnostics[f"{dataset}_{seed}"] = verify_teacher_and_graph(
                    original, teacher, simple_graph, x_cal, cfg, seed, device)
                if cfg.protocol=="rd":
                    diagnostics[f"{dataset}_{seed}"].update(verify_nested_method(simple_graph,cfg,seed))
                atomic_json(root / "verification.json", diagnostics)
                log(f"{dataset}, seed {seed}: folding, gauge, cycle bound, marginal gain and codec checks PASS")
            float_blob = encode_model(teacher.cpu(), data.architecture)
            teacher.to(device)
            denominator = sum(p.numel() for p in teacher.parameters())
            names = [a.name for a in simple_graph.layers]
            fp_key = condition_key(dataset, seed, "FP32", "FP32")
            if fp_key not in completed:
                measured = evaluate(teacher,test_loader,device,reference=teacher,temperature=cfg.kd_temperature)
                row = dict(dataset=dataset, seed=seed, method="FP32", budget="FP32",
                           accuracy=measured["accuracy"], initial_accuracy=measured["accuracy"],
                           test_loss=measured["loss"], bytes=len(float_blob),
                           teacher_kl=0.0,logit_nmse=0.0,initial_teacher_kl=0.0,initial_logit_nmse=0.0,
                           bits_per_parameter=len(float_blob)*8/denominator, compression_ratio=1.0,
                           compression_seconds=0.0, used_nonpot_fraction=0.0, reserved_fraction=0.0,
                           refinement_steps=0, selected_step=0, initial_validation_accuracy=teacher_info["best_validation_accuracy"],
                           selected_validation_accuracy=teacher_info["best_validation_accuracy"],
                           teacher_training_seconds=teacher_info["training_seconds"], smoke=cfg.smoke)
                rows.append(row); completed.add(fp_key); save_rows(rows, result_path)
            sketch_started = time.monotonic()
            sketch_result=sensitivity_sketch(teacher,simple_graph,x_cal,cfg,device,seed,return_vectors=cfg.protocol=="rd")
            if cfg.protocol=="rd":
                alpha,task_vectors=sketch_result
                check_graph=WeightGraph(teacher,data.architecture,alpha)
                check_graph.task_sketch=task_vectors
                if not cfg.skip_self_tests:
                    diagnostics[f"{dataset}_{seed}"].update(verify_sketch_geometry(check_graph,cfg,seed))
                    atomic_json(root/"verification.json",diagnostics)
                del check_graph
            else:
                alpha=sketch_result
            sketch_seconds = time.monotonic() - sketch_started
            allocation_cache={}
            # Every condition below is fixed in advance. Test scores never control
            # hyperparameters, extra masks, retries, or checkpoint selection.
            for method, budget, value in conditions:
                key = condition_key(dataset, seed, method, budget)
                if key in completed:
                    continue
                log(f"{dataset}, seed {seed}: {method}, {budget}")
                started = time.monotonic()
                info = {}
                if method in ("Uniform+KD","LSQ-W+KD"):
                    quantizer=lsq_spec if method=="LSQ-W+KD" else uniform_spec
                    specs = {a.name: quantizer(module_at(teacher, a.name).weight.detach().cpu().numpy(), int(value))
                             for a in simple_graph.layers}
                    base_model = teacher
                elif method == "PoT":
                    specs, _, _ = graph_specs(simple_graph, np.zeros(simple_graph.n_nodes),
                                              np.zeros(simple_graph.n_weights, bool), cfg.exponent_bits)
                    base_model = teacher
                elif method in (*RD_MAIN_METHODS,*RD_ABLATIONS):
                    graph=WeightGraph(teacher,data.architecture,alpha)
                    graph.task_sketch=task_vectors
                    scales,specs,info=allocate_guarded_rd(graph,int(value),method,cfg,x_cal,device,allocation_cache)
                    base_model=graph.apply(scales).to(device)
                else:
                    sensitivity = np.ones_like(alpha) if method == "CAPQ-no-sensitivity" else alpha
                    graph = WeightGraph(teacher, data.architecture, sensitivity)
                    scales, mask, info = allocate_terms(graph, float(value), method, cfg)
                    specs, _, _ = graph_specs(graph, scales, mask, cfg.exponent_bits)
                    base_model = graph.apply(scales).to(device)
                # SAME optimizer, number of calibration steps, and selection set.
                student = install_quantizers(base_model, specs).to(device)
                initial_model, initial_specs = materialize_student(student, data.architecture, names)
                initial_blob = encode_model(initial_model, data.architecture, initial_specs)
                initial_model = assert_roundtrip(initial_model, initial_blob)
                slug = f"{dataset}_s{seed}_{method}_{budget}".replace("+", "_").replace("=", "").replace(".", "_")
                selected_model, final_specs, refine_info = refine_student(
                    student, teacher, x_cal, y_cal, selection_loader, seed, cfg, device,
                    root / "refinement" / (slug + ".pt"), data.architecture, names)
                blob = encode_model(selected_model, data.architecture, final_specs)
                if "byte_cap" in info:
                    assert len(initial_blob)<=info["byte_cap"] and len(blob)<=info["byte_cap"], "Packed-byte cap exceeded"
                    assert len(blob)==int(encode_model(selected_model,data.architecture,final_specs,size_only=True))
                selected_model = assert_roundtrip(selected_model, blob)
                compression_seconds = time.monotonic() - started
                compression_seconds+=info.get("reused_guard_allocation_seconds",0.0)
                # Charge the common initialization as if this model were
                # compressed alone, even when the experiment reused it.
                compression_seconds+=info.get("reused_gauge_selection_seconds",0.0)
                if method not in ("Uniform+KD", "LSQ-W+KD", "PoT", "CAPQ-no-sensitivity"):
                    compression_seconds += sketch_seconds
                # Measure both pre-specified stages ONLY after refinement/selection
                # is complete. Neither test result feeds back into the algorithm.
                initial_test=evaluate(initial_model.to(device),test_loader,device,reference=teacher,temperature=cfg.kd_temperature)
                selected_test=evaluate(selected_model.to(device),test_loader,device,reference=teacher,temperature=cfg.kd_temperature)
                packed_dir = root / "packed_models"
                packed_dir.mkdir(exist_ok=True)
                (packed_dir / (slug + ".capq")).write_bytes(blob)
                reserved = used = 0
                for spec in final_specs.values():
                    if spec["kind"] == "pot":
                        reserved += int(spec["mask"].sum())
                        used += int(np.count_nonzero(spec["mask"] & ~np.isin(spec["tag"], [0, 4])))
                    elif spec["kind"] == "nested":
                        sizes=group_sizes(int(np.prod(spec["shape"])),spec["block_size"])
                        wdepth=np.repeat(spec["group_depths"],sizes)
                        reserved+=int((wdepth>0).sum())
                        used+=int((spec["mantissa"]!=0).sum())
                row = dict(dataset=dataset, seed=seed, method=method, budget=budget,
                           accuracy=selected_test["accuracy"], initial_accuracy=initial_test["accuracy"],
                           test_loss=selected_test["loss"], bytes=len(blob),
                           teacher_kl=selected_test["teacher_kl"],logit_nmse=selected_test["logit_nmse"],
                           initial_teacher_kl=initial_test["teacher_kl"],initial_logit_nmse=initial_test["logit_nmse"],
                           bits_per_parameter=len(blob)*8/denominator,
                           compression_ratio=len(float_blob)/len(blob), compression_seconds=compression_seconds,
                           used_nonpot_fraction=used/simple_graph.n_weights,
                           reserved_fraction=reserved/simple_graph.n_weights,
                           teacher_training_seconds=teacher_info["training_seconds"], smoke=cfg.smoke,
                           **info,
                           **refine_info)
                rows.append(row); completed.add(key); save_rows(rows, result_path)
                log(f"Measured test: initial={row['initial_accuracy']:.2f}%, selected={row['accuracy']:.2f}%; "
                    f"{len(blob):,} bytes, {row['bits_per_parameter']:.3f} bits/parameter, "
                    f"{row['compression_ratio']:.2f}× compression")
                del student, selected_model, initial_model
                if device.type == "cuda":
                    torch.cuda.empty_cache()
            del original, teacher, simple_graph
            if device.type == "cuda":
                torch.cuda.empty_cache()
        report_results(rows, cfg, root, final=False)
    expected_all = {condition_key(dataset, seed, method, budget) for dataset in cfg.datasets for seed in SEEDS
                    for method, budget in [("FP32", "FP32"), *[(m, b) for m, b, _ in conditions]]}
    assert expected_all <= completed, f"Missing experimental conditions: {expected_all - completed}"
    report_results(rows, cfg, root, final=True)
    atomic_json(root / "RUN_COMPLETE.json", dict(fingerprint=cfg.fingerprint(), seeds=SEEDS,
                datasets=cfg.datasets, conditions_completed=len(expected_all),
                completed_utc=time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
                all_test_results_measured=True, all_seed_groups_complete=True))


def parse_config() -> Config:
    parser = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    parser.add_argument("--protocol",default="rd",choices=("rd","legacy"),help="Byte-matched CAPQ-2.3 protocol or legacy extra-term fractions")
    parser.add_argument("--datasets", default="cifar100,speechcommands",
                        help="Comma-separated: cifar100, speechcommands, fashionmnist, digits")
    parser.add_argument("--out", default="capq_outputs")
    parser.add_argument("--data-root", default="capq_data")
    parser.add_argument("--device", default="auto", choices=("auto", "cpu", "cuda"))
    parser.add_argument("--workers", type=int, default=0)
    parser.add_argument("--cpu-threads", type=int, default=2)
    parser.add_argument("--epochs", type=int, default=None, help="Override teacher epochs for all selected datasets")
    parser.add_argument("--refine-steps", type=int, default=240)
    parser.add_argument("--calibration-size", type=int, default=768)
    parser.add_argument("--selection-size", type=int, default=1024)
    parser.add_argument("--fractions", default="0.02,0.10,0.25")
    parser.add_argument("--uniform-bits", default="4,5,6")
    parser.add_argument("--smoke", action="store_true", help="Five-seed real-data reduced integration test; NOT paper evidence")
    parser.add_argument("--skip-self-tests", action="store_true")
    parser.add_argument("--methods", default="", help="Optional comma-separated compression methods; always five seeds")
    parser.add_argument("--teacher-cache", default="", help="Explicit trusted CAPQ output directory with completed matching teachers")
    parser.add_argument("--rd-batch",type=int,default=48)
    parser.add_argument("--rd-shortlist",type=int,default=128)
    parser.add_argument("--rd-passes",type=int,default=2)
    parser.add_argument("--rd-swaps",type=int,default=24)
    parser.add_argument("--rd-sketch-mix",type=float,default=0.25)
    parser.add_argument("--no-calibration-guard",action="store_true")
    parser.add_argument("--no-adaptive-metric",action="store_true")
    args = parser.parse_args()
    datasets = tuple(a.strip() for a in args.datasets.split(",") if a.strip())
    if not datasets or len(set(datasets)) != len(datasets) or any(
            a not in ("cifar100", "speechcommands", "fashionmnist", "digits") for a in datasets):
        parser.error("Choose distinct supported real datasets")
    cfg = Config(protocol=args.protocol,datasets=datasets, fractions=tuple(float(a) for a in args.fractions.split(",")),
                 uniform_bits=tuple(int(a) for a in args.uniform_bits.split(",") if a),
                 out=args.out, data_root=args.data_root, device=args.device, workers=args.workers,
                 cpu_threads=args.cpu_threads, refinement_steps=args.refine_steps,
                 calibration_size=args.calibration_size, selection_size=args.selection_size,
                 smoke=args.smoke, skip_self_tests=args.skip_self_tests,
                 methods=tuple(a.strip() for a in args.methods.split(',') if a.strip()), teacher_cache=args.teacher_cache,
                 rd_batch=args.rd_batch,rd_shortlist=args.rd_shortlist,rd_passes=args.rd_passes,rd_swaps=args.rd_swaps,
                 rd_sketch_mix=args.rd_sketch_mix,rd_calibration_guard=not args.no_calibration_guard,
                 rd_adaptive_metric=not args.no_adaptive_metric)
    if args.epochs is not None:
        cfg.cifar_epochs = cfg.speech_epochs = cfg.fashion_epochs = cfg.digits_epochs = args.epochs
    if args.smoke:
        cfg.cifar_epochs = cfg.speech_epochs = cfg.fashion_epochs = cfg.digits_epochs = 2
        cfg.calibration_size = min(cfg.calibration_size, 128)
        cfg.selection_size = min(cfg.selection_size, 128)
        cfg.refinement_steps = min(cfg.refinement_steps, 8)
        cfg.eval_interval = 4
        cfg.sketch_batches = cfg.sketch_probes = 1
        cfg.gauge_steps = cfg.allocation_rounds = 2
    if cfg.workers < 0 or cfg.cpu_threads < 1 or cfg.batch_size < 1 or cfg.refinement_steps < 0:
        parser.error("Invalid runtime configuration")
    if min(cfg.cifar_epochs, cfg.speech_epochs, cfg.fashion_epochs, cfg.digits_epochs) < 1:
        parser.error("Teacher training needs at least one epoch")
    if not cfg.uniform_bits or any(not 2 <= b <= 8 for b in cfg.uniform_bits):
        parser.error("Uniform bit widths must be between two and eight")
    if len(set(cfg.uniform_bits)) != len(cfg.uniform_bits) or len(set(cfg.fractions)) != len(cfg.fractions):
        parser.error("Bit widths and term budgets must be distinct")
    if cfg.calibration_size < 1 or cfg.selection_size < 1:
        parser.error("Calibration and validation selection sets must be nonempty")
    if cfg.rd_batch<1 or cfg.rd_shortlist<cfg.rd_batch or cfg.rd_passes<1 or cfg.rd_swaps<0:
        parser.error("RD allocation needs positive passes and shortlist >= batch >= 1")
    if not 0<cfg.rd_sketch_mix<1:
        parser.error("RD sketch shrinkage must lie strictly between zero and one")
    supported_methods = {"PoT", "Uniform+KD","LSQ-W+KD", *MAIN_METHODS, *ABLATIONS,*RD_MAIN_METHODS,*RD_ABLATIONS}
    if len(set(cfg.methods)) != len(cfg.methods) or any(m not in supported_methods for m in cfg.methods):
        parser.error("Choose distinct supported method names")
    available={method for method,_,_ in condition_list(dataclasses.replace(cfg,methods=()))}
    if any(method not in available for method in cfg.methods):
        parser.error("A selected method is outside this protocol; use --protocol legacy for CAPQ-1.x methods")
    if cfg.protocol=="rd" and min(cfg.uniform_bits)<cfg.exponent_bits+1:
        parser.error("Byte-matched RD budgets must use at least four uniform bits with the default exponent width")
    return cfg


if __name__ == "__main__":
    # Configure determinism before the first CUDA context is created.
    os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
    with contextlib.ExitStack() as backend_context:
        if not torch.cuda.is_available() and not Path("/proc/cpuinfo").exists():
            # Sandboxed CPU hosts can lack the processor inventory required by
            # optional optimized convolution backends. Colab exposes cpuinfo.
            torch.backends.mkldnn.enabled = False
            backend_context.enter_context(torch.backends.nnpack.flags(enabled=False))
        run_experiments(parse_config())


In [ ]:
%run capq_spl.py


[14:40:58] CAPQ-2.3; device=cuda; seeds=(11, 23, 37, 53, 71); output=/content/capq_outputs/b9815f041300d525
[14:40:58] FULL CNN PAPER PROTOCOL
[14:40:58] Preparing real dataset: cifar100
[14:40:59] Downloading cifar-100-python.tar.gz: 4 / 161 MiB
[14:41:08] {"dataset": "cifar100", "train": 45000, "validation": 5000, "test": 10000, "split_seed": 2026, "reduced_data": false}
[14:41:40] cifar100, seed 11, epoch 1/80: train CE=3.9885, validation=12.30%, best=12.30%
[14:42:00] cifar100, seed 11, epoch 2/80: train CE=3.4355, validation=17.48%, best=17.48%
[14:42:21] cifar100, seed 11, epoch 3/80: train CE=3.0354, validation=26.40%, best=26.40%
[14:42:44] cifar100, seed 11, epoch 4/80: train CE=2.7224, validation=29.52%, best=29.52%
[14:43:19] cifar100, seed 11, epoch 5/80: train CE=2.5039, validation=30.36%, best=30.36%
[14:43:41] cifar100, seed 11, epoch 6/80: train CE=2.3530, validation=35.68%, best=35.68%
[14:44:02] cifar100, seed 11, epoch 7/80: train CE=2.2273, validation=31.42%, best=3